In [ ]:

# Libraries and global settings

import gc
import os
import subprocess
import time
import warnings
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.lines as mlines
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

# Hyperspectral and image processing
import spectral
import spectral.io.envi as envi
from skimage.filters import threshold_otsu
from skimage.morphology import binary_erosion, closing, disk

# Wavelength selection
from SpectralCARSLib import competitive_adaptive_sampling

# Global settings
SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

# Plotting defaults
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 200

# silence non-lowercase parameter warning from .hdr files
spectral.settings.envi_support_nonlowercase_params = True


# Hyperspectral Data Processing

In [ ]:
base = os.getcwd()

# bag definitions 
bags_train = {
    "A1": ["S1A1", "S2A1", "S3A1"], "A2": ["S1A2", "S2A2", "S3A2"], "A3": ["S1A3", "S2A3", "S3A3"],
    "B1": ["S1B1", "S2B1", "S3B1"], "B2": ["S1B2", "S2B2", "S3B2"], "B3": ["S1B3", "S2B3", "S3B3"],
    "B4": ["S1B4", "S2B4", "S3B4"], "C1": ["S1C1", "S2C1", "S3C1"], "C2": ["S1C2", "S2C2", "S3C2"],
    "C3": ["S1C3", "S2C3", "S3C3"], "D1": ["S1D1", "S2D1", "S3D1"], "D2": ["S1D2", "S2D2", "S3D2"],
    "D3": ["S1D3", "S2D3", "S3D3"], "E1": ["S1E1", "S2E1", "S3E1"], "E2": ["S1E2", "S2E2", "S3E2"],
    "E3": ["S1E3", "S2E3", "S3E3"], "E4": ["S1E4", "S2E4", "S3E4"], "F1": ["S1F1", "S2F1", "S3F1"],
    "F2": ["S1F2", "S2F2", "S3F2"], "F3": ["S1F3", "S2F3", "S3F3"], "F4": ["S1F4", "S2F4", "S3F4"],
    "G1": ["S1G1", "S2G1", "S3G1"], "G2": ["S1G2", "S2G2", "S3G2"], "G3": ["S1G3", "S2G3", "S3G3"],
    "G4": ["S1G4", "S2G4", "S3G4"], "H1": ["S1H1", "S2H1", "S3H1"], "H2": ["S1H2", "S2H2", "S3H2"],
    "H3": ["S1H3", "S2H3", "S3H3"], "H4": ["S1H4", "S2H4", "S3H4"],
}
bags_val = {
    "X1": ["S1X1", "S2X1", "S3X1"], "X2": ["S1X2", "S2X2", "S3X2"], "Z1": ["S1Z1", "S2Z1", "S3Z1"],
    "V1": ["S1V1", "S2V1", "S3V1"], "V2": ["S1V2", "S2V2", "S3V2"], "V3": ["S1V3", "S2V3", "S3V3"],
    "V4": ["S1V4", "S2V4", "S3V4"], "V5": ["S1V5", "S2V5", "S3V5"],
}
bags = {**bags_train, **bags_val}

# parameters 
crop     = 50    # columns removed from each side
mid_band = 112   

# one cube in memory at a time 
mean_spectra = {}
masks        = {}
clip_summary = []

for bag, replicates in bags.items():
    for sample_id in replicates:
        cap = os.path.join(base, f"LupinFlour_{sample_id}", "capture")

        # load + calibrate ONE capture
        sample = envi.open(os.path.join(cap, f"LupinFlour_{sample_id}.hdr")).load().astype(np.float32)
        dark   = envi.open(os.path.join(cap, f"DARKREF_LupinFlour_{sample_id}.hdr")).load().astype(np.float32)
        white  = envi.open(os.path.join(cap, f"WHITEREF_LupinFlour_{sample_id}.hdr")).load().astype(np.float32)

        dark_mean  = dark.mean(axis=0, keepdims=True)
        white_mean = white.mean(axis=0, keepdims=True)
        ref = (sample - dark_mean) / (white_mean - dark_mean)

        del sample, dark, white, dark_mean, white_mean
        gc.collect()

        # clipping report (unchanged behaviour)
        n_high = int((ref > 1).sum())
        n_low  = int((ref < 0).sum())
        if n_high + n_low > 0:
            pct = 100 * (n_high + n_low) / ref.size
            clip_summary.append((sample_id, n_high, n_low, pct))
        np.clip(ref, 0, 1, out=ref)

        # crop bright edge columns
        ref = ref[:, crop:-crop, :]

        # ROI mask: OTSU on the band-averaged grayscale, then morphological closing
        gray   = ref.mean(axis=2)
        thresh = threshold_otsu(gray)
        closed = closing(gray > thresh, disk(10))
        masks[sample_id] = closed
        del gray

        
        roi = ref[closed]                     # (n_roi, bands)
        mean_spectra[sample_id] = {
            "mean": roi.mean(axis=0),
            "std":  roi.std(axis=0),
        }

        del ref, roi, closed
        gc.collect()

# wavelengths: read from the header only (no cube load)
first_id = next(iter(mean_spectra))
hdr_obj  = envi.open(os.path.join(base, f"LupinFlour_{first_id}", "capture",
                                  f"LupinFlour_{first_id}.hdr"))
wavelengths = np.array(hdr_obj.metadata["Wavelength"], dtype=np.float32)

np.savez_compressed(
    os.path.join(base, "mean_spectra_cache.npz"),
    ids=np.array(list(mean_spectra.keys())),
    means=np.stack([mean_spectra[s]["mean"] for s in mean_spectra]),
    stds=np.stack([mean_spectra[s]["std"] for s in mean_spectra]),
    wavelengths=wavelengths,
)

print(f"{len(mean_spectra)} replicate spectra computed")
print(f"Wavelengths: {wavelengths[0]:.1f}-{wavelengths[-1]:.1f} nm ({len(wavelengths)} bands)")



In [ ]:
# Load-on-demand helper for the figure & map cells below.
base     = globals().get("base", os.getcwd())
crop     = globals().get("crop", 50)
mid_band = globals().get("mid_band", 112)

def _roi_mask_from_cube(cal_cropped_cube):
    gray = cal_cropped_cube.mean(axis=2)
    return closing(gray > threshold_otsu(gray), disk(10))

def load_cube_on_demand(sample_id, stage="calibrated_cropped"):
    
    cap = os.path.join(base, f"LupinFlour_{sample_id}", "capture")
    raw = envi.open(os.path.join(cap, f"LupinFlour_{sample_id}.hdr")).load().astype(np.float32)
    if stage == "raw":
        return raw
    dark  = envi.open(os.path.join(cap, f"DARKREF_LupinFlour_{sample_id}.hdr")).load().astype(np.float32)
    white = envi.open(os.path.join(cap, f"WHITEREF_LupinFlour_{sample_id}.hdr")).load().astype(np.float32)
    ref = (raw - dark.mean(axis=0, keepdims=True)) / (white.mean(axis=0, keepdims=True) - dark.mean(axis=0, keepdims=True))
    del raw, dark, white
    np.clip(ref, 0, 1, out=ref)
    if stage == "calibrated_uncropped":
        return ref
    ref = ref[:, crop:-crop, :]
    if stage == "calibrated_cropped":
        return ref
    if stage == "masked":
        m = masks[sample_id] if ("masks" in globals() and sample_id in masks) else _roi_mask_from_cube(ref)
        ref[~m] = np.nan          # 2-D ROI mask broadcast over all bands
        return ref
    raise ValueError(f"unknown stage: {stage!r}")


## Spectral Preprocessing
This section applies different preprocessing methods to the 21 mean replicate spectra before bag averaging and PLS modelling.

In [ ]:
# initialization 


# master dictionary to store all preprocessing versions
preprocessed_spectra = {}

# helper function: bag average
def bag_average(spectra_dict):
    result = {}
    for bag, replicates in bags.items():
        rep_means = np.stack([spectra_dict[sid] for sid in replicates])
        result[bag] = {
            "mean": rep_means.mean(axis=0),
            "std":  rep_means.std(axis=0)
        }
    return result

# helper function: plot bag spectra
def plot_bag_spectra(bag_spectra_dict, title, ylabel="Reflectance (0–1)"):
    fig, ax = plt.subplots(figsize=(12, 6))
    colours = plt.cm.tab10(np.linspace(0, 1, len(bags)))
    for (bag, spectra), colour in zip(bag_spectra_dict.items(), colours):
        ax.plot(wavelengths, spectra["mean"],
                color=colour, linewidth=1.5, label=bag)
        ax.fill_between(wavelengths,
                        spectra["mean"] - spectra["std"],
                        spectra["mean"] + spectra["std"],
                        color=colour, alpha=0.15)
    ax.set_title(title)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel(ylabel)
    ax.legend(title="Bag", fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# no preprocessing strategy 
# extract mean spectrum per sample_id (no preprocessing)
raw_spectra = {sid: mean_spectra[sid]["mean"] for sid in mean_spectra}

# bag average 
raw_bag_spectra = bag_average(raw_spectra)

# store
preprocessed_spectra["raw"] = raw_bag_spectra

# plot
print("Raw spectra stored in preprocessed_spectra['raw']:")
plot_bag_spectra(raw_bag_spectra, "No preprocessing")


In [ ]:
# standard normal variate (SNV)

def apply_snv(spectrum):
    # apply SNV to a single spectrum 
    return (spectrum - spectrum.mean()) / spectrum.std()

# apply SNV to all 21 replicate mean spectra
snv_spectra = {sid: apply_snv(mean_spectra[sid]["mean"]) for sid in mean_spectra}

# bag average
snv_bag_spectra = bag_average(snv_spectra)

# store 
preprocessed_spectra["snv"] = snv_bag_spectra

# plot
print("SNV spectra stored in preprocessed_spectra['snv']:")
plot_bag_spectra(snv_bag_spectra, "Standard Normal Variate (SNV)")

In [ ]:
# multiplicative scatter correction (MSC)

# MSC needs a reference spectrum (we can use the mean of all 21 spectra)
all_means = np.stack([mean_spectra[sid]["mean"] for sid in mean_spectra])
reference  = all_means.mean(axis=0)  # shape: (bands,)

def apply_msc(spectrum, reference):
    # apply MSC to a single spectrum using a reference spectrum
    # fit a linear regression: spectrum ≈ a + b * reference
    coeffs = np.polyfit(reference, spectrum, deg=1)  # returns [b, a]
    b, a   = coeffs
    # correct: remove additive (a) and multiplicative (b) effects
    return (spectrum - a)/b

# apply MSC to all 21 replicate mean spectra
msc_spectra = {sid: apply_msc(mean_spectra[sid]["mean"], reference) 
               for sid in mean_spectra}

# bag average 
msc_bag_spectra = bag_average(msc_spectra)

# store 
preprocessed_spectra["msc"] = msc_bag_spectra

# plot
print("MSC spectra stored in preprocessed_spectra['msc']:")
plot_bag_spectra(msc_bag_spectra, "Multiplicative Scatter Correction (MSC)")


In [ ]:
# savitzky-golay 1st and 2nd derivatives

# SG parameters
windows  = [9, 11, 15]  # window size (must be odd)
polyorder = 2  # polynomial order

for window in windows:
    # SG 1st derivative
    sg1_spectra = {sid: savgol_filter(mean_spectra[sid]["mean"],
                                      window_length=window,
                                      polyorder=polyorder,
                                      deriv=1)
                   for sid in mean_spectra}

    sg1_bag = bag_average(sg1_spectra)
    preprocessed_spectra[f"sg1_w{window}"] = sg1_bag
    print(f"SG 1st derivative window={window} stored in preprocessed_spectra['sg1_w{window}']:")
    plot_bag_spectra(sg1_bag,
                     f"SG 1st Derivative (window={window}, order={polyorder})")
    

    # SG 2nd derivative
    sg2_spectra = {sid: savgol_filter(mean_spectra[sid]["mean"],
                                      window_length=window,
                                      polyorder=polyorder,
                                      deriv=2)
                   for sid in mean_spectra}

    sg2_bag = bag_average(sg2_spectra)
    preprocessed_spectra[f"sg2_w{window}"] = sg2_bag
    print(f"SG 2nd derivative window={window} stored in preprocessed_spectra['sg2_w{window}']:")
    plot_bag_spectra(sg2_bag,
                     f"SG 2nd Derivative (window={window}, order={polyorder})")
    

print(f"\nAll SG versions stored: {[k for k in preprocessed_spectra.keys() if 'sg' in k]}")

In [ ]:
# combos

windows   = [9, 11, 15]
polyorder = 2

for window in windows:
    for deriv, deriv_name in [(1, "sg1"), (2, "sg2")]:

        # First apply SG derivative
        sg_spectra = {sid: savgol_filter(mean_spectra[sid]["mean"],
                                         window_length=window,
                                         polyorder=polyorder,
                                         deriv=deriv)
                      for sid in mean_spectra}
        
        # SG + SNV
        sg_snv_spectra = {sid: apply_snv(sg_spectra[sid])
                          for sid in sg_spectra}

        key = f"{deriv_name}_w{window}_snv"
        sg_snv_bag = bag_average(sg_snv_spectra)
        preprocessed_spectra[key] = sg_snv_bag
        print(f"SG + SNV combo stored in preprocessed_spectra['{key}']:")
        plot_bag_spectra(sg_snv_bag,
                         f"SG {deriv_name} (window={window}) + SNV")
        
        # SG + MSC
        sg_msc_spectra = {sid: apply_msc(sg_spectra[sid], reference)
                          for sid in sg_spectra}

        key = f"{deriv_name}_w{window}_msc"
        sg_msc_bag = bag_average(sg_msc_spectra)
        preprocessed_spectra[key] = sg_msc_bag
        print(f"SG + MSC combo stored in preprocessed_spectra['{key}']:")
        plot_bag_spectra(sg_msc_bag,
                         f"SG {deriv_name} (window={window}) + MSC")
    

In [ ]:
# quick check (edge wavelengths)
print("First 10 wavelengths:")
for i, w in enumerate(wavelengths[:10]):
    print(f"  Band {i}: {w:.1f} nm")

print("\nLast 10 wavelengths:")
for i, w in enumerate(wavelengths[-10:]):
    print(f"  Band {len(wavelengths)-10+i}: {w:.1f} nm")

In [ ]:
# removing bands 0-7 (up to 959.8 nm) and 218-223 (from 1702 nm onwards)

trim_left  = 8   # remove bands 0-7 (935.6-959.8 nm)
trim_right = 6   # remove bands 218-223 (1702.3-1720.2 nm)

# trim wavelengths array
wavelengths_trimmed = wavelengths[trim_left:-trim_right]
print(f"Wavelength range after trimming: "
      f"{wavelengths_trimmed[0]:.1f} to {wavelengths_trimmed[-1]:.1f} nm "
      f"({len(wavelengths_trimmed)} bands)")

# trim all preprocessing versions
preprocessed_trimmed = {}

for method, bag_spectra in preprocessed_spectra.items():
    preprocessed_trimmed[method] = {}
    for bag, spectra in bag_spectra.items():
        preprocessed_trimmed[method][bag] = {
            "mean": spectra["mean"][trim_left:-trim_right],
            "std":  spectra["std"][trim_left:-trim_right]
        }

print(f"\ncheck if all methods trimmed:")
for method in preprocessed_trimmed.keys():
    example = list(preprocessed_trimmed[method].values())[0]["mean"]
    print(f"  - {method}: {example.shape[0]} bands")

# before vs after trimming (example SG2+SNV w11)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colours = plt.cm.tab10(np.linspace(0, 1, len(bags)))

# before trimming
for (bag, spectra), colour in zip(preprocessed_spectra["sg2_w11_snv"].items(), colours):
    axes[0].plot(wavelengths, spectra["mean"],
                 color=colour, linewidth=1.5, label=bag)
axes[0].set_title("SG 2nd derivative + SNV (before trimming)")
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Units")
axes[0].legend(title="Bag", fontsize=8)
axes[0].grid(True, alpha=0.3)

# after trimming 
for (bag, spectra), colour in zip(preprocessed_trimmed["sg2_w11_snv"].items(), colours):
    axes[1].plot(wavelengths_trimmed, spectra["mean"],
                 color=colour, linewidth=1.5, label=bag)
axes[1].set_title("SG 2nd derivative + SNV (after trimming)")
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("Units")
axes[1].legend(title="Bag", fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Edge band trimming: before vs after (SG 2nd derivative + SNV, w11)",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# loading reference data
import pandas as pd 

excel_path = os.path.join(base,"Samples_LupinFlour.xlsx")

# load protein sheet
data_protein = pd.read_excel(excel_path, sheet_name="Protein")
data_protein.columns = ["sample_id", "protein_weight", "protein_pct", "protein_mean"]

# keep only rows where protein_mean is not NaN 
protein_means = data_protein[data_protein["protein_mean"].notna()][["sample_id", "protein_mean"]].copy()

# extract bag ID
protein_means["bag"] = protein_means["sample_id"].str[2:]  # remove "S1" prefix
protein_means = protein_means.set_index("bag")["protein_mean"]

print("Protein means per bag:")
print(protein_means)

# load moisture sheet
data_moisture = pd.read_excel(excel_path, sheet_name="Moisture")
data_moisture.columns = ["sample_id", "dish_weight", "total_weight",
                       "dessicator_weight", "sample_weight",
                       "moisture_pct", "moisture_mean"]

# keep only rows where moisture_mean is not NaN
moisture_means = data_moisture[data_moisture["moisture_mean"].notna()][["sample_id", "moisture_mean"]].copy()

# extract bag ID
moisture_means["bag"] = moisture_means["sample_id"].str[2:]
moisture_means = moisture_means.set_index("bag")["moisture_mean"]

print("Moisture means per bag:")
print(moisture_means)


# keep only info we have HSI data for 
available_bags = list(bags.keys())  # all bags with HSI data

protein_means  = protein_means[protein_means.index.isin(available_bags)]
moisture_means = moisture_means[moisture_means.index.isin(available_bags)]

print(f"\nAvailable bags with both HSI and reference data")
print(f"Bags: {list(protein_means.index)}")
print(f"Protein range: {protein_means.min():.2f}-{protein_means.max():.2f} %")
print(f"Moisture range: {moisture_means.min():.3f}-{moisture_means.max():.3f} %")

In [ ]:
# bag metadata: brand name and origin per brand
# H is the German brand (Rapunzel); V is the in-house lab flour.

brand_names = {
    "A": "Seara",
    "B": "Prozis",
    "C": "Markal",
    "D": "Diética",
    "E": "Priméal",
    "F": "Próvida",
    "G": "Più Bene",
    "H": "Rapunzel",
    "X": "Il Fruttarolo",
    "Z": "Bongiovanni",
    "V": "In-house",
}

origin_map = {
    "A": "Portugal",
    "B": "Unknown",      # not stated on Prozis packaging
    "C": "France",
    "D": "Portugal",
    "E": "France",
    "F": "Portugal",
    "G": "Italy",
    "H": "Germany",
    "X": "Italy",
    "Z": "Italy",
    "V": "N/A",          # in-house, not a sourced product
}

# all 37 bags are white lupin (Lupinus albus) 

# quick verification
assert set(brand_names) == set(origin_map), "brand_names and origin_map must cover the same brands"
print(f"Brands defined: {len(brand_names)}")
print(f"\n{'Brand':>5} | {'Name':<15} | {'Origin':<10}")
print("_" * 38)
for b in sorted(brand_names):
    print(f"{b:>5} | {brand_names[b]:<15} | {origin_map[b]:<10}")

In [ ]:
# apply lupin-specific protein conversion factor

OLD_CF = 6.25
NEW_CF = 5.44
correction_ratio = NEW_CF / OLD_CF

print(f"Applying lupin-specific protein conversion factor:")
print(f"  Original conversion factor: {OLD_CF} (default for general food matrices)")
print(f"  New conversion factor:      {NEW_CF} (lupin-specific)")

# apply to all protein values in the protein_means dict
protein_means_corrected = {}
for bag, value in protein_means.items():
    if value is None or (isinstance(value, float) and np.isnan(value)):
        protein_means_corrected[bag] = np.nan
    else:
        protein_means_corrected[bag] = value * correction_ratio

# overwrite protein_means with corrected values
protein_means = protein_means_corrected

# verify with a few examples
print(f"\nExample conversions:")
for bag in ["A1", "C1", "X1"]:
    if bag in protein_means:
        original = protein_means[bag] / correction_ratio  # back-calc original for display
        print(f"  {bag}: from {original:.2f}% (CF=6.25) to {protein_means[bag]:.2f}% (CF=5.44)")

In [ ]:
# building the spectral matrices and metadata table

# target variable: protein (moisture kept as metadata for characterization only)
# training set: brands A-G (generic brands)
# external validation set: brands X, Z (Italian brands and in-house flour, intentionally held out)

# consistent bag ordering
bag_order_train = list(bags_train.keys())
bag_order_val   = list(bags_val.keys())

# metadata DataFrame: one row per bag, to hold target + characterization info
# extra columns (brand, origin, variety, etc.) can be added later
metadata_train = pd.DataFrame({
    "bag":      bag_order_train,
    "set":      "train",
    "brand":    [bag[0]  for bag in bag_order_train],   # letter prefix (A, B, C, ...)
    "protein":  [protein_means[bag]  for bag in bag_order_train],
    "moisture": [moisture_means[bag] for bag in bag_order_train],
}).set_index("bag")

metadata_val = pd.DataFrame({
    "bag":      bag_order_val,
    "set":      "val",
    "brand":    [bag[0]  for bag in bag_order_val],
    "protein":  [protein_means[bag]  for bag in bag_order_val],
    "moisture": [moisture_means[bag] for bag in bag_order_val],
}).set_index("bag")

for df in (metadata_train, metadata_val):
    df["brand_name"] = df["brand"].map(brand_names)
    df["origin"]     = df["brand"].map(origin_map)

# target vector, protein only 
Y_train = metadata_train["protein"].values
Y_val   = metadata_val["protein"].values

# X matrices 
X_matrices_train = {}
X_matrices_val   = {}

for method, bag_spectra in preprocessed_trimmed.items():
    X_matrices_train[method] = np.array([bag_spectra[bag]["mean"] for bag in bag_order_train])
    X_matrices_val[method]   = np.array([bag_spectra[bag]["mean"] for bag in bag_order_val])

# summary
print("Training set metadata")
print(metadata_train.to_string())
print(f"\nExternal validation set metadata")
print(metadata_val.to_string())

print(f"\nX matrices (training) = {len(bag_order_train)} bags")
for method, X in X_matrices_train.items():
    print(f"  {method}: shape {X.shape}")

print(f"\nX matrices (external validation) = {len(bag_order_val)} bags")
for method, X in X_matrices_val.items():
    print(f"  {method}: shape {X.shape}")

print(f"\nWavelength axis: {wavelengths_trimmed[0]:.1f}-{wavelengths_trimmed[-1]:.1f} nm "
      f"({len(wavelengths_trimmed)} bands)")
print(f"\nTarget: protein")
print(f"  Training range:   {Y_train.min():.2f}-{Y_train.max():.2f} %  "
      f"(mean {Y_train.mean():.2f}, std {Y_train.std():.2f})")
print(f"  Validation range: {Y_val.min():.2f}-{Y_val.max():.2f} %  "
      f"(mean {Y_val.mean():.2f}, std {Y_val.std():.2f})")
print(f"  Training range:   {metadata_train['moisture'].min():.3f}-{metadata_train['moisture'].max():.3f} %")
print(f"  Validation range: {metadata_val['moisture'].min():.3f}-{metadata_val['moisture'].max():.3f} %")

In [ ]:
# verify which bags have complete reference data 

n_protein_train = metadata_train["protein"].notna().sum()
n_protein_val   = metadata_val["protein"].notna().sum()
n_moisture_train = metadata_train["moisture"].notna().sum()
n_moisture_val   = metadata_val["moisture"].notna().sum()

print(f"Reference data availability:")
print(f"  Training set ({len(metadata_train)} bags total):")
print(f"    Protein:  {n_protein_train}/{len(metadata_train)} bags")
print(f"    Moisture: {n_moisture_train}/{len(metadata_train)} bags")
print(f"  Validation set ({len(metadata_val)} bags total):")
print(f"    Protein:  {n_protein_val}/{len(metadata_val)} bags")
print(f"    Moisture: {n_moisture_val}/{len(metadata_val)} bags")

# flag missing bags
missing_protein_train = metadata_train[metadata_train["protein"].isna()].index.tolist()
missing_protein_val   = metadata_val[metadata_val["protein"].isna()].index.tolist()
missing_moisture_train = metadata_train[metadata_train["moisture"].isna()].index.tolist()
missing_moisture_val   = metadata_val[metadata_val["moisture"].isna()].index.tolist()

if missing_protein_train or missing_protein_val:
    if missing_protein_train: print(f"  Training: {missing_protein_train}")
    if missing_protein_val:   print(f"  Validation: {missing_protein_val}")

if missing_moisture_train or missing_moisture_val:
    if missing_moisture_train: print(f"  Training: {missing_moisture_train}")
    if missing_moisture_val:   print(f"  Validation: {missing_moisture_val}")

# Moisture Content Prediction

In [ ]:
# verify moisture coverage
print("Moisture values per bag (training set):")
for bag in bag_order_train:
    print(f"  {bag}: {metadata_train.loc[bag, 'moisture']:.3f}%")
print("\nMoisture values per bag (validation set):")
for bag in bag_order_val:
    print(f"  {bag}: {metadata_val.loc[bag, 'moisture']:.3f}%")
print(f"\nmissing values? Training: {metadata_train['moisture'].isna().sum()}, "
      f"Validation: {metadata_val['moisture'].isna().sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(metadata_train["moisture"], bins=10, color="C0", edgecolor="black",
        alpha=0.7, label=f"Training (n={len(metadata_train)})")
ax.hist(metadata_val["moisture"], bins=10, color="C3", edgecolor="black",
        alpha=0.7, label=f"Validation (n={len(metadata_val)})")
ax.set_xlabel("Moisture (%)")
ax.set_ylabel("Count")
ax.set_title("Moisture distribution across bags")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTraining: range {metadata_train['moisture'].min():.2f}-{metadata_train['moisture'].max():.2f}%, "
      f"std {metadata_train['moisture'].std():.3f}")
print(f"Validation: range {metadata_val['moisture'].min():.2f}-{metadata_val['moisture'].max():.2f}%, "
      f"std {metadata_val['moisture'].std():.3f}")

# Moisture prediction 

Y_train_moisture = metadata_train["moisture"].values
Y_val_moisture   = metadata_val["moisture"].values

print(f"Target: moisture (%)")
print(f"Training: n={len(Y_train_moisture)}, "
      f"range {Y_train_moisture.min():.2f}-{Y_train_moisture.max():.2f}, "
      f"mean {Y_train_moisture.mean():.2f}, std {Y_train_moisture.std():.3f}")
print(f"Validation: n={len(Y_val_moisture)}, "
      f"range {Y_val_moisture.min():.2f}-{Y_val_moisture.max():.2f}, "
      f"mean {Y_val_moisture.mean():.2f}, std {Y_val_moisture.std():.3f}")

# sanity: all validation values within training range
val_in_range = ((Y_val_moisture >= Y_train_moisture.min()) &
                (Y_val_moisture <= Y_train_moisture.max())).all()
print(f"\nAll validation values within training range: {val_in_range}")

In [ ]:
# Moisture preprocessing comparison - full sweep over all 21 preprocessing methods

max_lvs = 10
lv_range = range(1, max_lvs + 1)

# identify unique brands in training set
bag_ids = np.array(bag_order_train)
brands  = sorted(set(bag[0] for bag in bag_ids))

# storage: for each preprocessing, full CV results for moisture
cv_results_moisture = {}

# outer loop: preprocessing methods 
for method, X_full in X_matrices_train.items():
    Y_full = Y_train_moisture

    fold_rmse = np.zeros((len(brands), max_lvs))
    predictions_cv = {n_lv: np.full(len(bag_ids), np.nan) for n_lv in lv_range}

    # inner loop: CV folds
    for fold_idx, held_out in enumerate(brands):
        test_mask  = np.array([bag[0] == held_out for bag in bag_ids])
        train_mask = ~test_mask

        X_cal,  Y_cal  = X_full[train_mask], Y_full[train_mask]
        X_test, Y_test = X_full[test_mask],  Y_full[test_mask]

        for lv_idx, n_lv in enumerate(lv_range):
            model = PLSRegression(n_components=n_lv, scale=False)
            model.fit(X_cal, Y_cal)
            Y_pred = model.predict(X_test).ravel()

            fold_rmse[fold_idx, lv_idx] = np.sqrt(mean_squared_error(Y_test, Y_pred))
            predictions_cv[n_lv][test_mask] = Y_pred

    # aggregate
    rmsecv    = np.array([np.sqrt(mean_squared_error(Y_full, predictions_cv[n_lv]))
                          for n_lv in lv_range])
    rmsecv_se = fold_rmse.std(axis=0) / np.sqrt(len(brands))

    n_lv_min  = lv_range[int(np.argmin(rmsecv))]
    threshold = rmsecv.min() + rmsecv_se[np.argmin(rmsecv)]
    n_lv_1se  = lv_range[int(np.argmax(rmsecv <= threshold))]

    cv_results_moisture[method] = {
        "rmsecv":      rmsecv,
        "rmsecv_se":   rmsecv_se,
        "fold_rmse":   fold_rmse,
        "n_lv_min":    n_lv_min,
        "n_lv_1se":    n_lv_1se,
        "rmsecv_min":  rmsecv[n_lv_min - 1],
        "rmsecv_1se":  rmsecv[n_lv_1se - 1],
        "predictions": predictions_cv,
    }

print(f"Finished: {len(cv_results_moisture)} preprocessing methods × "
      f"{len(brands)} folds × {max_lvs} LVs\n")

# ranking table 
summary_moisture = pd.DataFrame({
    "method":      list(cv_results_moisture.keys()),
    "RMSECV_min":  [cv_results_moisture[m]["rmsecv_min"] for m in cv_results_moisture],
    "LV_min":      [cv_results_moisture[m]["n_lv_min"]   for m in cv_results_moisture],
    "RMSECV_1SE":  [cv_results_moisture[m]["rmsecv_1se"] for m in cv_results_moisture],
    "LV_1SE":      [cv_results_moisture[m]["n_lv_1se"]   for m in cv_results_moisture],
})
summary_moisture = summary_moisture.sort_values("RMSECV_min").reset_index(drop=True)

print("Ranking by minimum RMSECV (best → worst) - target: moisture\n")
print(summary_moisture.to_string(index=False,
                        formatters={"RMSECV_min": "{:.3f}".format,
                                    "RMSECV_1SE": "{:.3f}".format}))

# visual comparison 
top_n = 3
top_methods_moisture = summary_moisture.head(top_n)["method"].tolist()
print(f"\nTop {top_n} methods for moisture: {top_methods_moisture}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# left: all methods, top 3 highlighted
for method in cv_results_moisture:
    is_top = method in top_methods_moisture
    axes[0].plot(lv_range, cv_results_moisture[method]["rmsecv"],
                 color="C0" if is_top else "lightgrey",
                 linewidth=2 if is_top else 0.8,
                 alpha=1.0 if is_top else 0.5,
                 label=method if is_top else None)
axes[0].set_xlabel("Number of latent variables")
axes[0].set_ylabel("RMSECV (% moisture)")
axes[0].set_title(f"RMSECV vs LVs: all preprocessings (top {top_n} highlighted)")
axes[0].set_xticks(list(lv_range))
axes[0].grid(True, alpha=0.3)
axes[0].legend(title="Top methods")

# right: bar chart of minimum RMSECV per method, sorted
colours = ["C0" if m in top_methods_moisture else "lightgrey" for m in summary_moisture["method"]]
axes[1].barh(summary_moisture["method"], summary_moisture["RMSECV_min"], color=colours)
axes[1].invert_yaxis()
axes[1].set_xlabel("Minimum RMSECV (% moisture)")
axes[1].set_title("Ranking of preprocessing methods (moisture)")
axes[1].grid(True, alpha=0.3, axis="x")

plt.suptitle("PLSR - preprocessing comparison for MOISTURE (brand-level LOOCV)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Moisture diagnostic - winning preprocessing 

winner_moisture = "sg1_w11_msc"
n_lv_moisture   = cv_results_moisture[winner_moisture]["n_lv_min"]
rmsecv_moisture = cv_results_moisture[winner_moisture]["rmsecv_min"]

print(f"Winning preprocessing for moisture: {winner_moisture}")
print(f"Optimal latent variables: {n_lv_moisture}")
print(f"RMSECV: {rmsecv_moisture:.3f} % moisture\n")

# per-fold breakdown 
fold_rmse_at_lv = cv_results_moisture[winner_moisture]["fold_rmse"][:, n_lv_moisture - 1]
fold_summary_moisture = pd.DataFrame({
    "brand":  brands,
    "n_bags": [sum(bag[0] == b for bag in bag_order_train) for b in brands],
    "RMSE":   fold_rmse_at_lv,
    "moisture_mean":  [np.mean([metadata_train.loc[bag, "moisture"]
                                for bag in bag_order_train if bag[0] == b])
                       for b in brands],
}).sort_values("RMSE")

print("Per-fold RMSE (sorted):")
print(fold_summary_moisture.to_string(index=False,
                              formatters={"RMSE": "{:.3f}".format,
                                          "moisture_mean": "{:.3f}".format}))

# pooled CV predictions and metrics 
Y_cv_pred = cv_results_moisture[winner_moisture]["predictions"][n_lv_moisture]
Y_true    = Y_train_moisture
residuals = Y_true - Y_cv_pred

# fit global model for regression coefficient interpretation
model_full = PLSRegression(n_components=n_lv_moisture, scale=False)
model_full.fit(X_matrices_train[winner_moisture], Y_train_moisture)
coef_spectrum = model_full.coef_.ravel()

# summary metrics
ss_res = np.sum((Y_true - Y_cv_pred) ** 2)
ss_tot = np.sum((Y_true - Y_true.mean()) ** 2)
r2_cv  = 1 - ss_res / ss_tot
rpd    = Y_true.std() / rmsecv_moisture
bias   = residuals.mean()

print(f"\nCross-validated model performance:")
print(f"  RMSECV: {rmsecv_moisture:.3f} % moisture")
print(f"  R²:     {r2_cv:.3f}")
print(f"  RPD:    {rpd:.2f}   (>2 = useful, >2.5 = good, >3 = excellent for NIR)")
print(f"  Bias:   {bias:+.3f} % moisture")

# four-panel diagnostic plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

brand_colours = {b: plt.cm.tab10(i) for i, b in enumerate(brands)}
point_brands = [bag[0] for bag in bag_order_train]

# (1) predicted vs measured
for b in brands:
    mask = np.array([pb == b for pb in point_brands])
    axes[0, 0].scatter(Y_true[mask], Y_cv_pred[mask],
                       color=brand_colours[b], s=70, edgecolor="black",
                       linewidth=0.5, label=f"brand {b}")

y_min = min(Y_true.min(), Y_cv_pred.min()) - 0.3
y_max = max(Y_true.max(), Y_cv_pred.max()) + 0.3
axes[0, 0].plot([y_min, y_max], [y_min, y_max],
                color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0, 0].set_xlim(y_min, y_max)
axes[0, 0].set_ylim(y_min, y_max)
axes[0, 0].set_xlabel("Measured moisture (%)")
axes[0, 0].set_ylabel("Predicted moisture (%) [cross-validated]")
axes[0, 0].set_title(f"Predicted vs measured - {winner_moisture}, {n_lv_moisture} LVs\n"
                     f"R² = {r2_cv:.3f},  RMSECV = {rmsecv_moisture:.3f},  RPD = {rpd:.2f}")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8, loc="best")
axes[0, 0].set_aspect("equal", adjustable="box")

# (2) residuals vs measured
for b in brands:
    mask = np.array([pb == b for pb in point_brands])
    axes[0, 1].scatter(Y_true[mask], residuals[mask],
                       color=brand_colours[b], s=70, edgecolor="black",
                       linewidth=0.5, label=f"brand {b}")
axes[0, 1].axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0, 1].set_xlabel("Measured moisture (%)")
axes[0, 1].set_ylabel("Residual (measured - predicted)")
axes[0, 1].set_title("Residuals vs measured")
axes[0, 1].grid(True, alpha=0.3)

# (3) per-fold RMSE bar chart
fold_colours = [brand_colours[b] for b in fold_summary_moisture["brand"]]
axes[1, 0].bar(fold_summary_moisture["brand"], fold_summary_moisture["RMSE"],
               color=fold_colours, edgecolor="black", linewidth=0.5)
axes[1, 0].axhline(rmsecv_moisture, color="black", linestyle="--", linewidth=1,
                   alpha=0.6, label=f"RMSECV = {rmsecv_moisture:.3f}")
axes[1, 0].set_xlabel("Held-out brand")
axes[1, 0].set_ylabel("Fold RMSE (% moisture)")
axes[1, 0].set_title("RMSE per held-out brand")
axes[1, 0].grid(True, alpha=0.3, axis="y")
axes[1, 0].legend()

# (4) regression coefficient spectrum
axes[1, 1].plot(wavelengths_trimmed, coef_spectrum,
                color="C0", linewidth=1.2)
axes[1, 1].axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel("Wavelength (nm)")
axes[1, 1].set_ylabel("Regression coefficient")
axes[1, 1].set_title(f"PLS regression coefficients - fit on all training data, "
                     f"{n_lv_moisture} LVs")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f"Diagnostic: winning model for moisture ({winner_moisture})",
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Setup for the preprocessing × model sweep (moisture)



SEED = 42

# inputs
Y_full         = Y_train_moisture
bag_ids_full   = bag_order_train
brands_cv      = sorted(set(b[0] for b in bag_ids_full))   # ['A',...,'H']
preprocessings = list(X_matrices_train.keys())             # 21 methods

# brand-level fold masks 
fold_masks = {
    b: np.array([bag[0] == b for bag in bag_ids_full])
    for b in brands_cv
}

# generic brand-level LOOCV RMSE; can also return per-fold RMSEs for SE calc
def loocv_rmse(X, Y, fold_masks, model_factory, return_fold_rmses=False):
    preds = np.full(len(Y), np.nan)
    fold_rmses = []
    for test_mask in fold_masks.values():
        train_mask = ~test_mask
        m = model_factory()
        m.fit(X[train_mask], Y[train_mask])
        y_hat = m.predict(X[test_mask]).ravel()
        preds[test_mask] = y_hat
        fold_rmses.append(np.sqrt(np.mean((y_hat - Y[test_mask]) ** 2)))
    rmsecv = float(np.sqrt(np.nanmean((preds - Y) ** 2)))
    if return_fold_rmses:
        return rmsecv, np.array(fold_rmses)
    return rmsecv

def _se_from_fold_rmses(fold_rmses):
    """Standard error across LOOCV folds."""
    return float(np.std(fold_rmses, ddof=1) / np.sqrt(len(fold_rmses)))

sweep_results_moisture = pd.DataFrame(
    columns=["preprocessing", "model",
             "RMSECV_min", "params_min",
             "SE_min", "threshold_1se",
             "RMSECV_1se", "params_1se",
             "rule_differs", "time_s"]
)

print(f"Setup complete:")
print(f"  Bags:           {len(bag_ids_full)}")
print(f"  Brands (folds): {len(brands_cv)} -> {brands_cv}")
print(f"  Preprocessings: {len(preprocessings)}")
print(f"  Random seed:    {SEED}")

In [ ]:
# PLSR sweep over 21 preprocessings (moisture, brand-level LOOCV)


pls_lvs = list(range(1, 11))

def best_plsr(X, Y):
    results = []  # (n_lv, rmsecv, se)
    for n_lv in pls_lvs:
        if n_lv >= X.shape[0]:
            break
        rmsecv, folds = loocv_rmse(
            X, Y, fold_masks,
            lambda nlv=n_lv: PLSRegression(n_components=nlv, scale=False),
            return_fold_rmses=True,
        )
        results.append((n_lv, rmsecv, _se_from_fold_rmses(folds)))

    # min-RMSECV pick
    n_min, r_min, se_min = min(results, key=lambda t: t[1])
    threshold = r_min + se_min

    # 1-SE pick: fewest LVs whose RMSECV <= threshold
    one_se = [t for t in results if t[1] <= threshold]
    n_1se, r_1se, _ = min(one_se, key=lambda t: t[0])

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"n_components": n_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"n_components": n_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_plsr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PLSR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_moisture = sweep_results_moisture[sweep_results_moisture["model"] != "PLSR"]
sweep_results_moisture = pd.concat([sweep_results_moisture, new], ignore_index=True)

print(f"PLSR sweep done in {time.time()-t0:.1f} s")


In [ ]:
# PCR sweep over 21 preprocessings (moisture, brand-level LOOCV)


pcr_pcs = list(range(1, 11))

def best_pcr(X, Y):
    results = []
    for n_pc in pcr_pcs:
        if n_pc >= X.shape[0]:
            break
        def factory(np_=n_pc):
            return Pipeline([
                ("scale", StandardScaler(with_std=False)),  # mean-center
                ("pca",   PCA(n_components=np_, random_state=SEED)),
                ("lr",    LinearRegression()),
            ])
        rmsecv, folds = loocv_rmse(X, Y, fold_masks, factory, return_fold_rmses=True)
        results.append((n_pc, rmsecv, _se_from_fold_rmses(folds)))

    n_min, r_min, se_min = min(results, key=lambda t: t[1])
    threshold = r_min + se_min
    one_se = [t for t in results if t[1] <= threshold]
    n_1se, r_1se, _ = min(one_se, key=lambda t: t[0])

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"n_components": n_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"n_components": n_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_pcr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PCR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_moisture = sweep_results_moisture[sweep_results_moisture["model"] != "PCR"]
sweep_results_moisture = pd.concat([sweep_results_moisture, new], ignore_index=True)

print(f"PCR sweep done in {time.time()-t0:.1f} s")


In [ ]:
# SVMR sweep over 21 preprocessings (moisture, brand-level LOOCV)


svr_grid = list(product([0.1, 1, 10, 100, 1000], [1e-4, 1e-3, 1e-2, 1e-1, 1]))

def best_svmr(X, Y):
    results = []  # (C, gamma, rmsecv, se)
    for C, gamma in svr_grid:
        def factory(C_=C, g_=gamma):
            return Pipeline([
                ("scale", StandardScaler()),
                ("svr",   SVR(C=C_, gamma=g_, kernel="rbf")),
            ])
        rmsecv, folds = loocv_rmse(X, Y, fold_masks, factory, return_fold_rmses=True)
        results.append((C, gamma, rmsecv, _se_from_fold_rmses(folds)))

    C_min, g_min, r_min, se_min = min(results, key=lambda t: t[2])
    threshold = r_min + se_min
    one_se = [t for t in results if t[2] <= threshold]
    # simplest: lowest C, then lowest gamma
    C_1se, g_1se, r_1se, _ = min(one_se, key=lambda t: (t[0], t[1]))

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"C": C_min, "gamma": g_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"C": C_1se, "gamma": g_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_svmr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "SVMR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_moisture = sweep_results_moisture[sweep_results_moisture["model"] != "SVMR"]
sweep_results_moisture = pd.concat([sweep_results_moisture, new], ignore_index=True)

print(f"SVMR sweep done in {time.time()-t0:.1f} s")

In [ ]:
# Sweep summary: dissertation tables per model


def _fmt_params(p, model):
    if model in ("PLSR", "PCR"):
        return str(p["n_components"])
    if model == "SVMR":
        return f"C={p['C']}, γ={p['gamma']:g}"
    return str(p)

def model_table(df, model, top_n=None):
    """Return a ranked table for one model."""
    sub = (df[df["model"] == model]
           .sort_values("RMSECV_min")
           .reset_index(drop=True)
           .copy())

    sub["Rank"]           = sub.index + 1
    sub["Preprocessing"]  = sub["preprocessing"]
    sub["LVs/PCs/SVs"]    = sub.apply(lambda r: _fmt_params(r["params_min"], model), axis=1)
    sub["RMSECV (%)"]     = sub["RMSECV_min"].round(3)
    sub["1-SE thr (%)"]   = sub["threshold_1se"].round(3)
    sub["1-SE pick"]      = sub.apply(lambda r: _fmt_params(r["params_1se"], model), axis=1)
    sub["RMSECV 1-SE (%)"] = sub["RMSECV_1se"].round(3)
    sub["Diff?"]          = np.where(sub["rule_differs"], "✓", "")

    out = sub[["Rank", "Preprocessing", "LVs/PCs/SVs",
               "RMSECV (%)", "1-SE thr (%)",
               "1-SE pick", "RMSECV 1-SE (%)", "Diff?"]]
    return out.head(top_n) if top_n else out

models_run = sweep_results_moisture["model"].unique().tolist()
print(f"Models in sweep: {models_run}")

# Per-model winners (overall best by min RMSECV)
print(f"\n{'_'*90}")
print(f"Per-model winners (moisture) by minimum RMSECV")
print(f"{'_'*90}")
for m in models_run:
    sub = sweep_results_moisture[sweep_results_moisture["model"] == m]
    best = sub.sort_values("RMSECV_min").iloc[0]
    diff_marker = "  [1-SE picks simpler model]" if best["rule_differs"] else ""
    print(f"  {m:5s}  {best['preprocessing']:18s}  "
          f"RMSECV = {best['RMSECV_min']:.3f}  "
          f"min-params = {best['params_min']}  "
          f"1-SE-params = {best['params_1se']}{diff_marker}")

# Top combinations overall
print(f"\n{'_'*90}")
print(f"Top 10 (preprocessing, model) combinations by min RMSECV")
print(f"{'_'*90}")
top10 = sweep_results_moisture.sort_values("RMSECV_min").head(10).reset_index(drop=True)
for _, r in top10.iterrows():
    marker = " *" if r["rule_differs"] else "  "
    print(f"{marker}{r['model']:5s}  {r['preprocessing']:18s}  "
          f"RMSECV = {r['RMSECV_min']:.3f}  min = {r['params_min']}  "
          f"1-SE = {r['params_1se']}")
print("  (* = 1-SE rule picked a simpler model than minimum RMSECV)")

# Per-model: top 5 + full 21
for model in models_run:
    print(f"\n{'_'*90}\n{model} Top 5 preprocessings\n{'_'*90}")
    print(model_table(sweep_results_moisture, model, top_n=5).to_string(index=False))

    print(f"\n{'_'*90}\n{model} All 21 preprocessings\n{'_'*90}")
    print(model_table(sweep_results_moisture, model).to_string(index=False))

# Figures for Results Section

In [ ]:
# Figure for results section 3.1.1: raw vs calibrated reflectance 
diagnostic_id   = "S1A1"
raw_sample_diag = load_cube_on_demand(diagnostic_id, "raw")
before_crop     = load_cube_on_demand(diagnostic_id, "calibrated_uncropped")[:, :, mid_band]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(raw_sample_diag[:, :, mid_band], cmap="gray")
axes[0].set_title("(a)")
axes[0].set_xlabel("Spatial Dimension X")
axes[0].set_ylabel("Spatial Dimension Y")
plt.colorbar(im0, ax=axes[0], label="Raw DN value", fraction=0.025, pad=0.02)

im1 = axes[1].imshow(before_crop, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("(b) Calibrated reflectance")
axes[1].set_xlabel("Spatial Dimension X")
plt.colorbar(im1, ax=axes[1], label="Reflectance (0-1)", fraction=0.025, pad=0.02)

plt.tight_layout()
plt.savefig("fig_3-1-1_raw_calibrated_cropped.png", dpi=200, bbox_inches="tight")
plt.show()

del raw_sample_diag
gc.collect()


In [ ]:
# Figure for results section 3.1.1: calibrated vs cropped reflectance  
diagnostic_id   = "S1A1"
before_crop     = load_cube_on_demand(diagnostic_id, "calibrated_uncropped")[:, :, mid_band]
after_crop_img  = load_cube_on_demand(diagnostic_id, "calibrated_cropped")[:, :, mid_band]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(before_crop, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("(b)")
axes[0].set_xlabel("Spatial Dimension X")
axes[0].set_ylabel("Spatial Dimension Y")

im1 = axes[1].imshow(after_crop_img, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("(c)")
axes[1].set_xlabel("Spatial Dimension X")
plt.colorbar(im1, ax=axes[1], label="Reflectance", fraction=0.025, pad=0.02)

plt.tight_layout()
plt.savefig("fig_3-1-1b_calibrated_vs_cropped.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Figure for results section 3.1.2: ROI segmentation stages  
roi_id = "S1A1"

cube_diag   = load_cube_on_demand(roi_id, "calibrated_cropped")
gray_diag   = cube_diag.mean(axis=2)
thresh_diag = threshold_otsu(gray_diag)
binary_diag = gray_diag > thresh_diag
closed_diag = closing(binary_diag, disk(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(binary_diag, cmap="gray")
axes[0].set_title("(a)")
axes[0].set_ylabel("Spatial Dimension Y")
axes[0].set_xlabel("Spatial Dimension X")

axes[1].imshow(gray_diag, cmap="gray")
axes[1].contour(closed_diag, colors="red", linewidths=1.2)
axes[1].set_title("(b)")
axes[1].set_ylabel("Spatial Dimension Y")
axes[1].set_xlabel("Spatial Dimension X")

plt.tight_layout()
plt.savefig("fig_3-1-2a_roi_segmentation.png", dpi=200, bbox_inches="tight")
plt.show()

del cube_diag
gc.collect()


In [ ]:
# Figure for results section 3.1.2: mean reflectance spectra per bag


fig, ax = plt.subplots(figsize=(13, 6))
colours = plt.cm.tab10(np.linspace(0, 1, len(bags)))

for (bag, replicates), colour in zip(bags.items(), colours):
    for sample_id in replicates:
        mean = mean_spectra[sample_id]["mean"]
        ax.plot(wavelengths, mean,
                color=colour, alpha=0.7, linewidth=0.9, label=bag)

# one legend entry per bag
handles, labels = ax.get_legend_handles_labels()
unique = []
seen = set()
for h, l in zip(handles, labels):
    if l not in seen:
        unique.append((h, l))
        seen.add(l)

# put legend outside the plot to keep the spectra readable
ax.legend(*zip(*unique), title="Bag", fontsize=7, ncol=2,
          loc="center left", bbox_to_anchor=(1.01, 0.5))

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance (0-1)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_3-1-2b_mean_spectra_per_bag.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.3: comparison of SG 1st vs 2nd derivative
# both at window size 11. Shows the 37 bag-mean spectra, with bags coloured
# by brand and arranged into a brand-grouped legend for readability.


# brand-level base palette + per-bag shading
all_brands = sorted({bag[0] for bag in bags})  # ['A','B','C','D','E','F','G','H','V','X','Z']
base_colours = dict(zip(all_brands, cm.tab20(np.linspace(0, 1, len(all_brands)))))

# build a bag → colour mapping: each bag gets a shade of its brand's base colour
def shade(base_rgba, factor):
    """factor in (0.6, 1.0): smaller = darker, larger = lighter."""
    r, g, b, a = base_rgba
    return (r * factor, g * factor, b * factor, a)

bag_colours = {}
for brand in all_brands:
    brand_bags = sorted(bag for bag in bags if bag[0] == brand)
    # spread shading factor evenly between 0.65 and 1.0 across the brand's bags
    factors = np.linspace(0.65, 1.0, len(brand_bags))
    for bag, factor in zip(brand_bags, factors):
        bag_colours[bag] = shade(base_colours[brand], factor)

methods_to_show = [
    ("sg1_w11", "(c) SG 1st derivative (window=11)"),
    ("sg2_w11", "(d) SG 2nd derivative (window=11)"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (method_key, panel_title) in zip(axes, methods_to_show):
    bag_spectra = preprocessed_spectra[method_key]
    for bag, spectra in bag_spectra.items():
        ax.plot(wavelengths, spectra["mean"],
                color=bag_colours[bag], alpha=0.85, linewidth=0.9)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Derivative")
    ax.set_title(panel_title)
    ax.grid(True, alpha=0.3)

# brand-grouped legend: columns are brands, rows within column are bags
sorted_bags = sorted(bags.keys(), key=lambda b: (b[0], b[1:]))
legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.15), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.savefig("fig_3-1-3b_sg1_vs_sg2.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.3: comparison of SNV and MSC preprocessing


methods_to_show = [
    ("snv", "(a) SNV"),
    ("msc", "(b) MSC"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (method_key, panel_title) in zip(axes, methods_to_show):
    bag_spectra = preprocessed_spectra[method_key]
    for bag in sorted_bags:
        ax.plot(wavelengths, bag_spectra[bag]["mean"],
                color=bag_colours[bag], alpha=0.85, linewidth=0.9)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Scatter-corrected reflectance")
    ax.set_title(panel_title)
    ax.grid(True, alpha=0.3)

legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.15), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.savefig("fig_3-1-3a_snv_vs_msc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.3: SG 1st derivative followed by scatter correction


methods_to_show = [
    ("sg1_w11_msc", "(e) SG 1st derivative (window=11) + MSC"),
    ("sg1_w11_snv", "(f) SG 1st derivative (window=11) + SNV"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (method_key, panel_title) in zip(axes, methods_to_show):
    bag_spectra = preprocessed_spectra[method_key]
    for bag in sorted_bags:
        ax.plot(wavelengths, bag_spectra[bag]["mean"],
                color=bag_colours[bag], alpha=0.85, linewidth=0.9)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Scatter-corrected derivative")
    ax.set_title(panel_title)
    ax.grid(True, alpha=0.3)

legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.15), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.savefig("fig_3-1-3c_sg1_msc_vs_sg1_snv.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.3: SG 2nd derivative followed by scatter correction


methods_to_show = [
    ("sg2_w11_msc", "(g) SG 2nd derivative (window=11) + MSC"),
    ("sg2_w11_snv", "(h) SG 2nd derivative (window=11) + SNV"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (method_key, panel_title) in zip(axes, methods_to_show):
    bag_spectra = preprocessed_spectra[method_key]
    for bag in sorted_bags:
        ax.plot(wavelengths, bag_spectra[bag]["mean"],
                color=bag_colours[bag], alpha=0.85, linewidth=0.9)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Scatter-corrected derivative")
    ax.set_title(panel_title)
    ax.grid(True, alpha=0.3)

legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.15), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.savefig("fig_3-1-3d_sg2_msc_vs_sg2_snv.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.4: example of edge-band trimming before vs after


example_method = "sg2_w11_snv"
example_title  = "SG 2nd derivative (window=11) + SNV"

untrimmed_dict = preprocessed_spectra[example_method]
trimmed_dict   = preprocessed_trimmed[example_method]

# wavelength axes for the two versions
trim_left  = 8
trim_right = 6
wavelengths_trimmed = wavelengths[trim_left:-trim_right]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) untrimmed: 224 bands, 935.6-1720.2 nm
for bag in sorted_bags:
    axes[0].plot(wavelengths, untrimmed_dict[bag]["mean"],
                 color=bag_colours[bag], alpha=0.85, linewidth=0.9)
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Scatter-corrected derivative")
axes[0].set_title(f"(a)")
axes[0].grid(True, alpha=0.3)

# shade the regions that will be removed
axes[0].axvspan(wavelengths[0], wavelengths[trim_left - 1],
                color="red", alpha=0.12, label=f"Removed bands")
axes[0].axvspan(wavelengths[-trim_right], wavelengths[-1],
                color="red", alpha=0.12)
axes[0].legend(loc="upper right", fontsize=8, frameon=True)

# (b) trimmed: 210 bands, 963.0-1699.3 nm
for bag in sorted_bags:
    axes[1].plot(wavelengths_trimmed, trimmed_dict[bag]["mean"],
                 color=bag_colours[bag], alpha=0.85, linewidth=0.9)
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_title(f"(b)")
axes[1].grid(True, alpha=0.3)


axes[1].set_xlim(axes[0].get_xlim())

# shared bag-coloured legend at the bottom
legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.15), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig("fig_3-1-4_edge_trimming_example.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure for results section 3.1.1: raw DN spectra vs calibrated reflectance spectra

raw_mean_spectra = {}

for bag, replicates in bags.items():
    for sample_id in replicates:
        cap = os.path.join(base, f"LupinFlour_{sample_id}", "capture")
        raw = envi.open(os.path.join(cap, f"LupinFlour_{sample_id}.hdr")).load().astype(np.float32)

        # mirror the crop applied in Cell 4 (50 columns from each side)
        raw_cropped = raw[:, crop:-crop, :]

        # apply the same ROI mask used for the calibrated cubes
        mask_3d = masks[sample_id][:, :, np.newaxis]
        mask_3d = np.repeat(mask_3d, raw_cropped.shape[2], axis=2)
        raw_masked = raw_cropped.astype(np.float32)
        raw_masked[~mask_3d] = np.nan

        # spatial average across the ROI
        n_rows, n_cols, n_bands = raw_masked.shape
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            mean_raw = np.nanmean(raw_masked.reshape(n_rows * n_cols, n_bands), axis=0)

        raw_mean_spectra[sample_id] = mean_raw

        del raw, raw_cropped, raw_masked
        gc.collect()

print(f"Computed raw mean spectra for {len(raw_mean_spectra)} samples")

# average the three replicate spectra per bag, for both raw and calibrated 
raw_bag_means = {}
cal_bag_means = {}
for bag, replicates in bags.items():
    raw_stack = np.stack([raw_mean_spectra[sid] for sid in replicates])
    cal_stack = np.stack([mean_spectra[sid]["mean"] for sid in replicates])
    raw_bag_means[bag] = raw_stack.mean(axis=0)
    cal_bag_means[bag] = cal_stack.mean(axis=0)


fig, axes = plt.subplots(2, 1, figsize=(11, 11))

# (a) raw DN spectra
for bag in sorted_bags:
    axes[0].plot(wavelengths, raw_bag_means[bag],
                 color=bag_colours[bag], alpha=0.85, linewidth=0.9)
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Raw DN value")
axes[0].set_title("(a) Raw spectra (before calibration)")
axes[0].grid(True, alpha=0.3)

# (b) calibrated reflectance spectra
for bag in sorted_bags:
    axes[1].plot(wavelengths, cal_bag_means[bag],
                 color=bag_colours[bag], alpha=0.85, linewidth=0.9)
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("Reflectance (0-1)")
axes[1].set_title("(b) Reflectance spectra (after calibration)")
axes[1].grid(True, alpha=0.3)

# bag-listed legend
legend_handles = [
    mlines.Line2D([], [], color=bag_colours[bag], linewidth=2.0, label=bag)
    for bag in sorted_bags
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.06), ncol=len(all_brands), fontsize=8,
           frameon=True, title="Bag", columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.savefig("fig_3-1-1c_raw_vs_calibrated_spectra.png", dpi=200, bbox_inches="tight")
plt.show()

## CARS Wavelength Selection: Joint Preprocessing × Model Sweep
MOISTURE ONLY

In [ ]:
# CARS-based preprocessing × model sweep (moisture)


# CARS hyperparameters 
CARS_N_SAMPLINGS = 50    # Monte Carlo sampling runs inside one CARS call
CARS_MAX_LVS     = 10    # max latent variables CARS considers
CARS_CV_FOLDS    = 10     # CARS's own internal CV (internal to CARS)


sweep_results_moisture_cars = pd.DataFrame(
    columns=["preprocessing", "model", "RMSECV", "params", "n_bands_mean", "time_s"]
)

# helper: run CARS on a training matrix, return selected band indices
def run_cars_select(X_train, Y_train, seed=SEED):
    """Run CARS once and return the indices of selected wavelengths."""
    np.random.seed(seed)   # make the per-fold sweep selection reproducible
    out = competitive_adaptive_sampling(
        X=X_train,
        y=Y_train,
        max_components=CARS_MAX_LVS,
        folds=CARS_CV_FOLDS,
        iterations=CARS_N_SAMPLINGS,  
        adaptive_resampling=True,  
        preprocess="center",
        verbose=0,
    )
    return np.asarray(out["selected_variables"], dtype=int)

# LOOCV with CARS inside each fold 
def loocv_rmse_with_cars(X, Y, fold_masks, model_factory):
    preds = np.full(len(Y), np.nan)
    n_bands_list = []
    for test_mask in fold_masks.values():
        train_mask = ~test_mask
        # CARS fit on training brands only
        sel = run_cars_select(X[train_mask], Y[train_mask], seed=SEED)
        if len(sel) < 2:
            continue
        n_bands_list.append(len(sel))
        # fit model on CARS-selected bands of training data
        X_tr = X[train_mask][:, sel]
        X_te = X[test_mask][:, sel]
        max_safe = min(X_tr.shape[0] - 1, X_tr.shape[1])
        m = model_factory()
        if isinstance(m, Pipeline):
            for step in m.named_steps.values():
                if hasattr(step, "n_components"):
                    if step.n_components > max_safe:
                        step.n_components = max(1, max_safe)
        elif hasattr(m, "n_components"):
            if m.n_components > max_safe:
                m.n_components = max(1, max_safe)
        m.fit(X_tr, Y[train_mask])
        preds[test_mask] = m.predict(X_te).ravel()
    rmsecv = float(np.sqrt(np.nanmean((preds - Y) ** 2)))
    mean_bands = float(np.mean(n_bands_list)) if n_bands_list else float("nan")
    return rmsecv, mean_bands

print(f"  CARS hyperparameters: {CARS_N_SAMPLINGS} samplings, "
      f"{CARS_MAX_LVS} max LVs, {CARS_CV_FOLDS}-fold internal CV")
print(f"  Outer LOOCV: brand-level, {len(fold_masks)} folds")
print(f"  Random seed: {SEED}")

In [ ]:
# PLSR + CARS sweep over 21 preprocessings (moisture, brand-level LOOCV)


def best_plsr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for n_lv in pls_lvs:
        if n_lv >= X.shape[0]:
            break
        r, n_b = loocv_rmse_with_cars(
            X, Y, fold_masks,
            lambda nlv=n_lv: PLSRegression(n_components=nlv, scale=False),
        )
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"n_components": n_lv}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_plsr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PLSR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_moisture_cars = sweep_results_moisture_cars[
    sweep_results_moisture_cars["model"] != "PLSR"
]
sweep_results_moisture_cars = pd.concat(
    [sweep_results_moisture_cars, new], ignore_index=True
)

print(f"\nPLSR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# PCR + CARS sweep over 21 preprocessings (moisture, brand-level LOOCV)


def best_pcr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for n_pc in pcr_pcs:
        if n_pc >= X.shape[0]:
            break
        def factory(np_=n_pc):
            return Pipeline([
                ("scale", StandardScaler(with_std=False)),
                ("pca",   PCA(n_components=np_, random_state=SEED)),
                ("lr",    LinearRegression()),
            ])
        r, n_b = loocv_rmse_with_cars(X, Y, fold_masks, factory)
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"n_components": n_pc}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_pcr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PCR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_moisture_cars = sweep_results_moisture_cars[
    sweep_results_moisture_cars["model"] != "PCR"
]
sweep_results_moisture_cars = pd.concat(
    [sweep_results_moisture_cars, new], ignore_index=True
)

print(f"\nPCR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# SVMR + CARS sweep over 21 preprocessings (moisture, brand-level LOOCV)


def best_svmr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for C, gamma in svr_grid:
        def factory(C_=C, g_=gamma):
            return Pipeline([
                ("scale", StandardScaler()),
                ("svr",   SVR(C=C_, gamma=g_, kernel="rbf")),
            ])
        r, n_b = loocv_rmse_with_cars(X, Y, fold_masks, factory)
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"C": C, "gamma": gamma}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_svmr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "SVMR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_moisture_cars = sweep_results_moisture_cars[
    sweep_results_moisture_cars["model"] != "SVMR"
]
sweep_results_moisture_cars = pd.concat(
    [sweep_results_moisture_cars, new], ignore_index=True
)

print(f"\nSVMR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# Print CARS sweep tables for the dissertation


# format helper
def fmt_params(params):
    if "n_components" in params:
        return str(params["n_components"])
    if "C" in params and "gamma" in params:
        # match the formatting used in your full-spectrum table
        return f"C={params['C']}, γ={params['gamma']:.0e}".replace("e-0", "e-")
    return str(params)

def fmt_prep(prep):
    if prep == "raw":
        return "Raw"
    if prep == "snv":
        return "SNV"
    if prep == "msc":
        return "MSC"
    parts = prep.split("_")
    out = parts[0].upper()                          # SG1 or SG2
    out += f"({parts[1][1:]})"                       # (11)
    if len(parts) == 3:
        out += f"-{parts[2].upper()}"                # -MSC or -SNV
    return out

df = sweep_results_moisture_cars.copy()
df["Preprocessing"]   = df["preprocessing"].apply(fmt_prep)
df["Hyperparameters"] = df["params"].apply(fmt_params)
df["Mean Bands"]      = df["n_bands_mean"].round(0).astype("Int64")
df["RMSECV (%)"]      = df["RMSECV"].round(3)

MODEL_ORDER = ["PLSR", "PCR", "SVMR"]

#top 5 per model
print("_" * 90)
print("IN-TEXT TABLE: Top 5 (preprocessing, hyperparameters) per model")
print("_" * 90)
for m in MODEL_ORDER:
    sub = (df[df["model"] == m]
           .sort_values("RMSECV")
           .head(5)
           .reset_index(drop=True))
    sub.index = sub.index + 1
    sub.index.name = "Rank"
    print(f"\n{m}")
    print(sub[["Preprocessing", "Hyperparameters", "Mean Bands", "RMSECV (%)"]]
          .to_string())

#full 21 × 3 grid
print("\n\n" + "_" * 90)
print("APPENDIX TABLE: Full grid (21 preprocessings × 3 models)")
print("_" * 90)
for m in MODEL_ORDER:
    sub = (df[df["model"] == m]
           .sort_values("RMSECV")
           .reset_index(drop=True))
    sub.index = sub.index + 1
    sub.index.name = "Rank"
    print(f"\n{m}")
    print(sub[["Preprocessing", "Hyperparameters", "Mean Bands", "RMSECV (%)"]]
          .to_string())

## External Prediction
MOISTURE ONLY


In [ ]:
# External prediction - Full spectral range




# derive rank-1 winners from sweep_results_moisture (uses RMSECV_min rule)
sweep_winners_moisture = {}
for m in sweep_results_moisture["model"].unique():
    row = (sweep_results_moisture[sweep_results_moisture["model"] == m]
           .sort_values("RMSECV_min").iloc[0])
    sweep_winners_moisture[m] = {
        "preprocessing": row["preprocessing"],
        "RMSECV":        row["RMSECV_min"],
        "params":        row["params_min"],
    }

print("Full-spectrum rank-1 winners:")
for m, w in sweep_winners_moisture.items():
    print(f"  {m:5s}  prep={w['preprocessing']:18s}  "
          f"RMSECV={w['RMSECV']:.3f}  params={w['params']}")
print()

# model factories
def make_plsr(n_components):
    return PLSRegression(n_components=n_components, scale=False)

def make_pcr(n_components):
    return Pipeline([
        ("scale", StandardScaler(with_std=False)),
        ("pca",   PCA(n_components=n_components, random_state=SEED)),
        ("lr",    LinearRegression()),
    ])

def make_svmr(C, gamma):
    return Pipeline([
        ("scale", StandardScaler()),
        ("svr",   SVR(C=C, gamma=gamma, kernel="rbf")),
    ])

def evaluate(y_true, y_pred):
    rmsep = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    r2p   = float(r2_score(y_true, y_pred))
    bias  = float(np.mean(y_pred - y_true))
    return rmsep, r2p, bias

results_full = []
preds_full   = {}

for m_name, winner in sweep_winners_moisture.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    if m_name == "PLSR":
        model = make_plsr(params["n_components"])
    elif m_name == "PCR":
        model = make_pcr(params["n_components"])
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr, Y_train_moisture)
    y_pred = model.predict(X_va).ravel()
    preds_full[m_name] = y_pred

    rmsep, r2p, bias = evaluate(Y_val_moisture, y_pred)
    results_full.append({
        "model":          m_name,
        "analysis":       "Full",
        "preprocessing":  prep,
        "params":         params,
        "n_bands":        X_tr.shape[1],
        "RMSECV":         winner["RMSECV"],
        "RMSEP":          rmsep,
        "R2p":            r2p,
        "bias":           bias,
    })
    print(f"  {m_name:5s}  RMSEP={rmsep:.3f}  R²p={r2p:.3f}  bias={bias:+.3f}")

results_full_df = pd.DataFrame(results_full)
print()
print(results_full_df[["model", "preprocessing", "n_bands",
                       "RMSECV", "RMSEP", "R2p", "bias"]].to_string(index=False))

In [ ]:
# external prediction on CARS-selected wavelengths

sweep_winners_moisture_cars = {}
for m in sweep_results_moisture_cars["model"].unique():
    row = (sweep_results_moisture_cars[sweep_results_moisture_cars["model"] == m].sort_values("RMSECV").iloc[0])
    sweep_winners_moisture_cars[m] = {
        "preprocessing": row["preprocessing"],
        "RMSECV":        row["RMSECV"],
        "params":        row["params"],
        "n_bands_mean":  row["n_bands_mean"],
    }

print("CARS rank-1 winners (preprocessing per model):")
for m, w in sweep_winners_moisture_cars.items():
    print(f"  {m:5s}  prep={w['preprocessing']:14s}  params={w['params']}")
print()

results_cars = []
preds_cars   = {}
cars_subsets = {}

for m_name, winner in sweep_winners_moisture_cars.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    # one fixed-seed CARS run on the full calibration set
    sel = run_cars_select(X_tr, Y_train_moisture, seed=SEED)
    cars_subsets[m_name] = sel
    print(f"  {m_name:5s}  prep={prep:14s}  bands={len(sel)} -> "
          f"{', '.join(f'{wavelengths_trimmed[b]:.0f}' for b in sorted(sel))} nm")

    X_tr_sel = X_tr[:, sel]
    X_va_sel = X_va[:, sel]

    if m_name == "PLSR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model = make_plsr(n_comp)
    elif m_name == "PCR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model = make_pcr(n_comp)
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr_sel, Y_train_moisture)
    y_pred = model.predict(X_va_sel).ravel()
    preds_cars[m_name] = y_pred

    rmsep, r2p, bias = evaluate(Y_val_moisture, y_pred)
    results_cars.append({
        "model":         m_name,
        "analysis":      "CARS",
        "preprocessing": prep,
        "params":        params,
        "n_bands":       len(sel),
        "RMSECV":        winner["RMSECV"],
        "RMSEP":         rmsep,
        "R2p":           r2p,
        "bias":          bias,
    })
    print(f"  -> RMSEP={rmsep:.3f}  R²p={r2p:.3f}  bias={bias:+.3f}\n")

results_cars_df = pd.DataFrame(results_cars)
print("_" * 70)
print(results_cars_df[["model", "preprocessing", "n_bands",
                       "RMSECV", "RMSEP", "R2p", "bias"]].to_string(index=False))

# keep a target-specific copy so the wavelength cell can read both later
cars_subsets_moisture = dict(cars_subsets)


In [ ]:
# External prediction summary: moisture


combined = pd.concat([results_full_df, results_cars_df], ignore_index=True)
combined = combined.sort_values(["model", "analysis"]).reset_index(drop=True)

print("_" * 100)
print("External prediction summary: moisture")
print("_" * 100)
print(combined[["model", "analysis", "preprocessing", "n_bands",
                "RMSECV", "RMSEP", "R2p", "bias"]].to_string(
    index=False,
    formatters={"RMSECV": "{:.3f}".format,
                "RMSEP":  "{:.3f}".format,
                "R2p":    "{:.3f}".format,
                "bias":   "{:+.3f}".format},
))

print()
print("_" * 100)
print("Per-bag predictions on external set (V/X/Z bags)")
print("_" * 100)
per_bag = pd.DataFrame({"bag": bag_order_val, "y_true": Y_val_moisture})
for m in ["PLSR", "PCR", "SVMR"]:
    per_bag[f"{m}_Full"] = preds_full[m]
    per_bag[f"{m}_CARS"] = preds_cars[m]
print(per_bag.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
# Per-brand RMSEP and bias for all 6 configurations
brand_of = [b[0] for b in bag_order_val]
rows = []
for m in ["PLSR", "PCR", "SVMR"]:
    for analysis, preds in [("Full", preds_full[m]), ("CARS", preds_cars[m])]:
        for brand in ["V", "X", "Z"]:
            mask = np.array([br == brand for br in brand_of])
            y_t = Y_val_moisture[mask]
            y_p = preds[mask]
            rmsep = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
            bias  = float(np.mean(y_p - y_t))
            r2    = float(r2_score(y_t, y_p)) if mask.sum() >= 3 else float("nan")
            rows.append({"model": m, "analysis": analysis, "brand": brand,
                         "n_bags": int(mask.sum()),
                         "RMSEP": rmsep, "bias": bias, "R2p": r2})

per_brand = pd.DataFrame(rows)
print(per_brand.to_string(
    index=False,
    formatters={"RMSEP": "{:.3f}".format,
                "bias":  "{:+.3f}".format,
                "R2p":   "{:.3f}".format},
))

In [ ]:
# Complete NIR reporting metrics for all 6 configurations (moisture)

# RPD reference: SD of the prediction set 
sd_val = float(np.std(Y_val_moisture, ddof=1))

# R2cv computed from sweep results: R2cv = 1 - (RMSECV^2) / Var(Y_cal)
var_cal = float(np.var(Y_train_moisture, ddof=1))

rows = []

# Full-spectrum configurations
for m_name, winner in sweep_winners_moisture.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    if m_name == "PLSR":
        model = make_plsr(params["n_components"])
    elif m_name == "PCR":
        model = make_pcr(params["n_components"])
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr, Y_train_moisture)
    y_cal_pred = model.predict(X_tr).ravel()
    y_val_pred = model.predict(X_va).ravel()

    rmsec  = float(np.sqrt(np.mean((y_cal_pred - Y_train_moisture)**2)))
    r2c    = float(r2_score(Y_train_moisture, y_cal_pred))
    rmsecv = winner["RMSECV"]
    r2cv   = 1.0 - (rmsecv**2) / var_cal
    rmsep  = float(np.sqrt(np.mean((y_val_pred - Y_val_moisture)**2)))
    r2p    = float(r2_score(Y_val_moisture, y_val_pred))
    bias   = float(np.mean(y_val_pred - Y_val_moisture))
    rpd    = sd_val / rmsep

    rows.append({"model": m_name, "analysis": "Full",
                 "R2c": r2c, "RMSEC": rmsec,
                 "R2cv": r2cv, "RMSECV": rmsecv,
                 "R2p": r2p, "RMSEP": rmsep,
                 "bias": bias, "RPD": rpd})

# CARS configurations 
for m_name, winner in sweep_winners_moisture_cars.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    sel = cars_subsets[m_name]
    X_tr_sel = X_tr[:, sel]
    X_va_sel = X_va[:, sel]

    if m_name == "PLSR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model  = make_plsr(n_comp)
    elif m_name == "PCR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model  = make_pcr(n_comp)
    elif m_name == "SVMR":
        model  = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr_sel, Y_train_moisture)
    y_cal_pred = model.predict(X_tr_sel).ravel()
    y_val_pred = model.predict(X_va_sel).ravel()

    rmsec  = float(np.sqrt(np.mean((y_cal_pred - Y_train_moisture)**2)))
    r2c    = float(r2_score(Y_train_moisture, y_cal_pred))
    rmsecv = winner["RMSECV"]
    r2cv   = 1.0 - (rmsecv**2) / var_cal
    rmsep  = float(np.sqrt(np.mean((y_val_pred - Y_val_moisture)**2)))
    r2p    = float(r2_score(Y_val_moisture, y_val_pred))
    bias   = float(np.mean(y_val_pred - Y_val_moisture))
    rpd    = sd_val / rmsep

    rows.append({"model": m_name, "analysis": "CARS",
                 "R2c": r2c, "RMSEC": rmsec,
                 "R2cv": r2cv, "RMSECV": rmsecv,
                 "R2p": r2p, "RMSEP": rmsep,
                 "bias": bias, "RPD": rpd})

metrics_df = pd.DataFrame(rows).sort_values(["model","analysis"]).reset_index(drop=True)

print("_" * 110)
print("Complete NIR metrics: moisture")
print(f"SD(Y_val) = {sd_val:.3f} %  |  SD(Y_cal) = {np.std(Y_train_moisture, ddof=1):.3f} %")
print("_" * 110)
print(metrics_df.to_string(
    index=False,
    formatters={"R2c":    "{:.3f}".format,
                "RMSEC":  "{:.3f}".format,
                "R2cv":   "{:.3f}".format,
                "RMSECV": "{:.3f}".format,
                "R2p":    "{:.3f}".format,
                "RMSEP":  "{:.3f}".format,
                "bias":   "{:+.3f}".format,
                "RPD":    "{:.2f}".format},
))

# Protein Content Prediction

Protein-content pipeline mirrors the moisture analysis above; cells to be added once protein reference values are finalized.

In [ ]:
# verify protein coverage
print("Protein values per bag (training set):")
for bag in bag_order_train:
    print(f"  {bag}: {metadata_train.loc[bag, 'protein']:.3f}%")
print("\nProtein values per bag (validation set):")
for bag in bag_order_val:
    print(f"  {bag}: {metadata_val.loc[bag, 'protein']:.3f}%")
print(f"\nmissing values? Training: {metadata_train['protein'].isna().sum()}, "
      f"Validation: {metadata_val['protein'].isna().sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(metadata_train["protein"], bins=10, color="C0", edgecolor="black",
        alpha=0.7, label=f"Training (n={len(metadata_train)})")
ax.hist(metadata_val["protein"], bins=10, color="C3", edgecolor="black",
        alpha=0.7, label=f"Validation (n={len(metadata_val)})")
ax.set_xlabel("Protein (%)")
ax.set_ylabel("Count")
ax.set_title("Protein distribution across bags")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTraining: range {metadata_train['protein'].min():.2f}-{metadata_train['protein'].max():.2f}%, "
      f"std {metadata_train['protein'].std():.3f}")
print(f"Validation: range {metadata_val['protein'].min():.2f}-{metadata_val['protein'].max():.2f}%, "
      f"std {metadata_val['protein'].std():.3f}")

# Protein prediction - set up target variable


Y_train_protein = metadata_train["protein"].values
Y_val_protein   = metadata_val["protein"].values

print(f"Target: protein (%)")
print(f"Training: n={len(Y_train_protein)}, "
      f"range {Y_train_protein.min():.2f}-{Y_train_protein.max():.2f}, "
      f"mean {Y_train_protein.mean():.2f}, std {Y_train_protein.std():.3f}")
print(f"Validation: n={len(Y_val_protein)}, "
      f"range {Y_val_protein.min():.2f}-{Y_val_protein.max():.2f}, "
      f"mean {Y_val_protein.mean():.2f}, std {Y_val_protein.std():.3f}")

# sanity: all validation values within training range
val_in_range = ((Y_val_protein >= Y_train_protein.min()) &
                (Y_val_protein <= Y_train_protein.max())).all()
print(f"\nAll validation values within training range: {val_in_range}")

In [ ]:
# Protein preprocessing comparison - full sweep over all 21 preprocessing methods

max_lvs = 10
lv_range = range(1, max_lvs + 1)

# identify unique brands in training set
bag_ids = np.array(bag_order_train)
brands  = sorted(set(bag[0] for bag in bag_ids))

# storage: for each preprocessing, full CV results for protein
cv_results_protein = {}

# outer loop: preprocessing methods 
for method, X_full in X_matrices_train.items():
    Y_full = Y_train_protein

    fold_rmse = np.zeros((len(brands), max_lvs))
    predictions_cv = {n_lv: np.full(len(bag_ids), np.nan) for n_lv in lv_range}

    # inner loop: CV folds
    for fold_idx, held_out in enumerate(brands):
        test_mask  = np.array([bag[0] == held_out for bag in bag_ids])
        train_mask = ~test_mask

        X_cal,  Y_cal  = X_full[train_mask], Y_full[train_mask]
        X_test, Y_test = X_full[test_mask],  Y_full[test_mask]

        for lv_idx, n_lv in enumerate(lv_range):
            model = PLSRegression(n_components=n_lv, scale=False)
            model.fit(X_cal, Y_cal)
            Y_pred = model.predict(X_test).ravel()

            fold_rmse[fold_idx, lv_idx] = np.sqrt(mean_squared_error(Y_test, Y_pred))
            predictions_cv[n_lv][test_mask] = Y_pred

    # aggregate
    rmsecv    = np.array([np.sqrt(mean_squared_error(Y_full, predictions_cv[n_lv]))
                          for n_lv in lv_range])
    rmsecv_se = fold_rmse.std(axis=0) / np.sqrt(len(brands))

    n_lv_min  = lv_range[int(np.argmin(rmsecv))]
    threshold = rmsecv.min() + rmsecv_se[np.argmin(rmsecv)]
    n_lv_1se  = lv_range[int(np.argmax(rmsecv <= threshold))]

    cv_results_protein[method] = {
        "rmsecv":      rmsecv,
        "rmsecv_se":   rmsecv_se,
        "fold_rmse":   fold_rmse,
        "n_lv_min":    n_lv_min,
        "n_lv_1se":    n_lv_1se,
        "rmsecv_min":  rmsecv[n_lv_min - 1],
        "rmsecv_1se":  rmsecv[n_lv_1se - 1],
        "predictions": predictions_cv,
    }

print(f"Finished: {len(cv_results_protein)} preprocessing methods × "
      f"{len(brands)} folds × {max_lvs} LVs\n")

# ranking table 
summary_protein = pd.DataFrame({
    "method":      list(cv_results_protein.keys()),
    "RMSECV_min":  [cv_results_protein[m]["rmsecv_min"] for m in cv_results_protein],
    "LV_min":      [cv_results_protein[m]["n_lv_min"]   for m in cv_results_protein],
    "RMSECV_1SE":  [cv_results_protein[m]["rmsecv_1se"] for m in cv_results_protein],
    "LV_1SE":      [cv_results_protein[m]["n_lv_1se"]   for m in cv_results_protein],
})
summary_protein = summary_protein.sort_values("RMSECV_min").reset_index(drop=True)

print("Ranking by minimum RMSECV (best to worst) - target: protein\n")
print(summary_protein.to_string(index=False,
                        formatters={"RMSECV_min": "{:.3f}".format,
                                    "RMSECV_1SE": "{:.3f}".format}))

# visual comparison 
top_n = 3
top_methods_protein = summary_protein.head(top_n)["method"].tolist()
print(f"\nTop {top_n} methods for protein: {top_methods_protein}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# left: all methods, top 3 highlighted
for method in cv_results_protein:
    is_top = method in top_methods_protein
    axes[0].plot(lv_range, cv_results_protein[method]["rmsecv"],
                 color="C0" if is_top else "lightgrey",
                 linewidth=2 if is_top else 0.8,
                 alpha=1.0 if is_top else 0.5,
                 label=method if is_top else None)
axes[0].set_xlabel("Number of latent variables")
axes[0].set_ylabel("RMSECV (% protein)")
axes[0].set_title(f"RMSECV vs LVs: all preprocessings (top {top_n} highlighted)")
axes[0].set_xticks(list(lv_range))
axes[0].grid(True, alpha=0.3)
axes[0].legend(title="Top methods")

# right: bar chart of minimum RMSECV per method, sorted
colours = ["C0" if m in top_methods_protein else "lightgrey" for m in summary_protein["method"]]
axes[1].barh(summary_protein["method"], summary_protein["RMSECV_min"], color=colours)
axes[1].invert_yaxis()
axes[1].set_xlabel("Minimum RMSECV (% protein)")
axes[1].set_title("Ranking of preprocessing methods (protein)")
axes[1].grid(True, alpha=0.3, axis="x")

plt.suptitle("PLSR - preprocessing comparison for PROTEIN (brand-level LOOCV)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Protein diagnostic - winning preprocessing 

winner_protein = "sg2_w9_snv"  # from the ranking table above
n_lv_protein   = cv_results_protein[winner_protein]["n_lv_min"]
rmsecv_protein = cv_results_protein[winner_protein]["rmsecv_min"]

print(f"Winning preprocessing for protein: {winner_protein}")
print(f"Optimal latent variables: {n_lv_protein}")
print(f"RMSECV: {rmsecv_protein:.3f} % protein\n")

# per-fold breakdown 
fold_rmse_at_lv = cv_results_protein[winner_protein]["fold_rmse"][:, n_lv_protein - 1]
fold_summary_protein = pd.DataFrame({
    "brand":  brands,
    "n_bags": [sum(bag[0] == b for bag in bag_order_train) for b in brands],
    "RMSE":   fold_rmse_at_lv,
    "protein_mean":  [np.mean([metadata_train.loc[bag, "protein"]
                                for bag in bag_order_train if bag[0] == b])
                       for b in brands],
}).sort_values("RMSE")

print("Per-fold RMSE (sorted):")
print(fold_summary_protein.to_string(index=False,
                              formatters={"RMSE": "{:.3f}".format,
                                          "protein_mean": "{:.3f}".format}))

# pooled CV predictions and metrics 
Y_cv_pred = cv_results_protein[winner_protein]["predictions"][n_lv_protein]
Y_true    = Y_train_protein
residuals = Y_true - Y_cv_pred

# fit global model for regression coefficient interpretation
model_full = PLSRegression(n_components=n_lv_protein, scale=False)
model_full.fit(X_matrices_train[winner_protein], Y_train_protein)
coef_spectrum = model_full.coef_.ravel()

# summary metrics
ss_res = np.sum((Y_true - Y_cv_pred) ** 2)
ss_tot = np.sum((Y_true - Y_true.mean()) ** 2)
r2_cv  = 1 - ss_res / ss_tot
rpd    = Y_true.std() / rmsecv_protein
bias   = residuals.mean()

print(f"\nCross-validated model performance:")
print(f"  RMSECV: {rmsecv_protein:.3f} % protein")
print(f"  R²:     {r2_cv:.3f}")
print(f"  RPD:    {rpd:.2f}   (>2 = useful, >2.5 = good, >3 = excellent for NIR)")
print(f"  Bias:   {bias:+.3f} % protein")

# four-panel diagnostic plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

brand_colours = {b: plt.cm.tab10(i) for i, b in enumerate(brands)}
point_brands = [bag[0] for bag in bag_order_train]

# (1) predicted vs measured
for b in brands:
    mask = np.array([pb == b for pb in point_brands])
    axes[0, 0].scatter(Y_true[mask], Y_cv_pred[mask],
                       color=brand_colours[b], s=70, edgecolor="black",
                       linewidth=0.5, label=f"brand {b}")

y_min = min(Y_true.min(), Y_cv_pred.min()) - 0.3
y_max = max(Y_true.max(), Y_cv_pred.max()) + 0.3
axes[0, 0].plot([y_min, y_max], [y_min, y_max],
                color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0, 0].set_xlim(y_min, y_max)
axes[0, 0].set_ylim(y_min, y_max)
axes[0, 0].set_xlabel("Measured protein (%)")
axes[0, 0].set_ylabel("Predicted protein (%) [cross-validated]")
axes[0, 0].set_title(f"Predicted vs measured - {winner_protein}, {n_lv_protein} LVs\n"
                     f"R² = {r2_cv:.3f},  RMSECV = {rmsecv_protein:.3f},  RPD = {rpd:.2f}")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8, loc="best")
axes[0, 0].set_aspect("equal", adjustable="box")

# (2) residuals vs measured
for b in brands:
    mask = np.array([pb == b for pb in point_brands])
    axes[0, 1].scatter(Y_true[mask], residuals[mask],
                       color=brand_colours[b], s=70, edgecolor="black",
                       linewidth=0.5, label=f"brand {b}")
axes[0, 1].axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0, 1].set_xlabel("Measured protein (%)")
axes[0, 1].set_ylabel("Residual (measured - predicted)")
axes[0, 1].set_title("Residuals vs measured")
axes[0, 1].grid(True, alpha=0.3)

# (3) per-fold RMSE bar chart
fold_colours = [brand_colours[b] for b in fold_summary_protein["brand"]]
axes[1, 0].bar(fold_summary_protein["brand"], fold_summary_protein["RMSE"],
               color=fold_colours, edgecolor="black", linewidth=0.5)
axes[1, 0].axhline(rmsecv_protein, color="black", linestyle="--", linewidth=1,
                   alpha=0.6, label=f"RMSECV = {rmsecv_protein:.3f}")
axes[1, 0].set_xlabel("Held-out brand")
axes[1, 0].set_ylabel("Fold RMSE (% protein)")
axes[1, 0].set_title("RMSE per held-out brand")
axes[1, 0].grid(True, alpha=0.3, axis="y")
axes[1, 0].legend()

# (4) regression coefficient spectrum
axes[1, 1].plot(wavelengths_trimmed, coef_spectrum,
                color="C0", linewidth=1.2)
axes[1, 1].axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel("Wavelength (nm)")
axes[1, 1].set_ylabel("Regression coefficient")
axes[1, 1].set_title(f"PLS regression coefficients - fit on all training data, "
                     f"{n_lv_protein} LVs")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f"Diagnostic: winning model for protein ({winner_protein})",
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Setup for the preprocessing × model sweep (protein)


SEED = 42

# inputs
Y_full         = Y_train_protein
bag_ids_full   = bag_order_train
brands_cv      = sorted(set(b[0] for b in bag_ids_full))   # ['A',...,'H']
preprocessings = list(X_matrices_train.keys())             # 21 methods

# brand-level fold masks (precomputed)
fold_masks = {
    b: np.array([bag[0] == b for bag in bag_ids_full])
    for b in brands_cv
}

# generic brand-level LOOCV RMSE; can also return per-fold RMSEs for SE calc
def loocv_rmse(X, Y, fold_masks, model_factory, return_fold_rmses=False):
    preds = np.full(len(Y), np.nan)
    fold_rmses = []
    for test_mask in fold_masks.values():
        train_mask = ~test_mask
        m = model_factory()
        m.fit(X[train_mask], Y[train_mask])
        y_hat = m.predict(X[test_mask]).ravel()
        preds[test_mask] = y_hat
        fold_rmses.append(np.sqrt(np.mean((y_hat - Y[test_mask]) ** 2)))
    rmsecv = float(np.sqrt(np.nanmean((preds - Y) ** 2)))
    if return_fold_rmses:
        return rmsecv, np.array(fold_rmses)
    return rmsecv

def _se_from_fold_rmses(fold_rmses):
    """Standard error across LOOCV folds."""
    return float(np.std(fold_rmses, ddof=1) / np.sqrt(len(fold_rmses)))

# results accumulator (per-model cells append to this)
sweep_results_protein = pd.DataFrame(
    columns=["preprocessing", "model",
             "RMSECV_min", "params_min",
             "SE_min", "threshold_1se",
             "RMSECV_1se", "params_1se",
             "rule_differs", "time_s"]
)

print(f"Setup complete:")
print(f"  Bags:           {len(bag_ids_full)}")
print(f"  Brands (folds): {len(brands_cv)} -> {brands_cv}")
print(f"  Preprocessings: {len(preprocessings)}")
print(f"  Random seed:    {SEED}")

In [ ]:
# PLSR sweep over 21 preprocessings (protein, brand-level LOOCV)

pls_lvs = list(range(1, 11))

def best_plsr(X, Y):
    results = []  # (n_lv, rmsecv, se)
    for n_lv in pls_lvs:
        if n_lv >= X.shape[0]:
            break
        rmsecv, folds = loocv_rmse(
            X, Y, fold_masks,
            lambda nlv=n_lv: PLSRegression(n_components=nlv, scale=False),
            return_fold_rmses=True,
        )
        results.append((n_lv, rmsecv, _se_from_fold_rmses(folds)))

    # min-RMSECV pick
    n_min, r_min, se_min = min(results, key=lambda t: t[1])
    threshold = r_min + se_min

    # 1-SE pick: fewest LVs whose RMSECV <= threshold
    one_se = [t for t in results if t[1] <= threshold]
    n_1se, r_1se, _ = min(one_se, key=lambda t: t[0])

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"n_components": n_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"n_components": n_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_plsr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PLSR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_protein = sweep_results_protein[sweep_results_protein["model"] != "PLSR"]
sweep_results_protein = pd.concat([sweep_results_protein, new], ignore_index=True)

print(f"PLSR sweep done in {time.time()-t0:.1f} s")

In [ ]:
# PCR sweep over 21 preprocessings (protein, brand-level LOOCV)

pcr_pcs = list(range(1, 11))

def best_pcr(X, Y):
    results = []
    for n_pc in pcr_pcs:
        if n_pc >= X.shape[0]:
            break
        def factory(np_=n_pc):
            return Pipeline([
                ("scale", StandardScaler(with_std=False)),  # mean-center
                ("pca",   PCA(n_components=np_, random_state=SEED)),
                ("lr",    LinearRegression()),
            ])
        rmsecv, folds = loocv_rmse(X, Y, fold_masks, factory, return_fold_rmses=True)
        results.append((n_pc, rmsecv, _se_from_fold_rmses(folds)))

    n_min, r_min, se_min = min(results, key=lambda t: t[1])
    threshold = r_min + se_min
    one_se = [t for t in results if t[1] <= threshold]
    n_1se, r_1se, _ = min(one_se, key=lambda t: t[0])

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"n_components": n_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"n_components": n_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_pcr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PCR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_protein = sweep_results_protein[sweep_results_protein["model"] != "PCR"]
sweep_results_protein = pd.concat([sweep_results_protein, new], ignore_index=True)

print(f"PCR sweep done in {time.time()-t0:.1f} s")

In [ ]:
# SVMR sweep over 21 preprocessings (protein, brand-level LOOCV)

svr_grid = list(product([0.1, 1, 10, 100, 1000], [1e-4, 1e-3, 1e-2, 1e-1, 1]))

def best_svmr(X, Y):
    results = []  # (C, gamma, rmsecv, se)
    for C, gamma in svr_grid:
        def factory(C_=C, g_=gamma):
            return Pipeline([
                ("scale", StandardScaler()),
                ("svr",   SVR(C=C_, gamma=g_, kernel="rbf")),
            ])
        rmsecv, folds = loocv_rmse(X, Y, fold_masks, factory, return_fold_rmses=True)
        results.append((C, gamma, rmsecv, _se_from_fold_rmses(folds)))

    C_min, g_min, r_min, se_min = min(results, key=lambda t: t[2])
    threshold = r_min + se_min
    one_se = [t for t in results if t[2] <= threshold]
    # simplest: lowest C, then lowest gamma
    C_1se, g_1se, r_1se, _ = min(one_se, key=lambda t: (t[0], t[1]))

    return {
        "RMSECV_min":    r_min,
        "params_min":    {"C": C_min, "gamma": g_min},
        "SE_min":        se_min,
        "threshold_1se": threshold,
        "RMSECV_1se":    r_1se,
        "params_1se":    {"C": C_1se, "gamma": g_1se},
    }

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_svmr(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "SVMR",
        **out,
        "rule_differs":  out["params_min"] != out["params_1se"],
        "time_s":        round(time.time() - t_s, 2),
    })

new = pd.DataFrame(rows)
sweep_results_protein = sweep_results_protein[sweep_results_protein["model"] != "SVMR"]
sweep_results_protein = pd.concat([sweep_results_protein, new], ignore_index=True)

print(f"SVMR sweep done in {time.time()-t0:.1f} s")

In [ ]:
# Sweep summary: tables per model

def _fmt_params(p, model):
    if model in ("PLSR", "PCR"):
        return str(p["n_components"])
    if model == "SVMR":
        return f"C={p['C']}, γ={p['gamma']:g}"
    return str(p)

def model_table(df, model, top_n=None):
    """Return a ranked table for one model."""
    sub = (df[df["model"] == model]
           .sort_values("RMSECV_min")
           .reset_index(drop=True)
           .copy())

    sub["Rank"]           = sub.index + 1
    sub["Preprocessing"]  = sub["preprocessing"]
    sub["LVs/PCs/SVs"]    = sub.apply(lambda r: _fmt_params(r["params_min"], model), axis=1)
    sub["RMSECV (%)"]     = sub["RMSECV_min"].round(3)
    sub["1-SE thr (%)"]   = sub["threshold_1se"].round(3)
    sub["1-SE pick"]      = sub.apply(lambda r: _fmt_params(r["params_1se"], model), axis=1)
    sub["RMSECV 1-SE (%)"] = sub["RMSECV_1se"].round(3)
    sub["Diff?"]          = np.where(sub["rule_differs"], "✓", "")

    out = sub[["Rank", "Preprocessing", "LVs/PCs/SVs",
               "RMSECV (%)", "1-SE thr (%)",
               "1-SE pick", "RMSECV 1-SE (%)", "Diff?"]]
    return out.head(top_n) if top_n else out

models_run = sweep_results_protein["model"].unique().tolist()
print(f"Models in sweep: {models_run}")

# Per-model winners (overall best by min RMSECV)
print(f"\n{'_'*90}")
print(f"Per-model winners (protein) by minimum RMSECV")
print(f"{'_'*90}")
for m in models_run:
    sub = sweep_results_protein[sweep_results_protein["model"] == m]
    best = sub.sort_values("RMSECV_min").iloc[0]
    diff_marker = "  [1-SE picks simpler model]" if best["rule_differs"] else ""
    print(f"  {m:5s}  {best['preprocessing']:18s}  "
          f"RMSECV = {best['RMSECV_min']:.3f}  "
          f"min-params = {best['params_min']}  "
          f"1-SE-params = {best['params_1se']}{diff_marker}")

# Top combinations overall
print(f"\n{'_'*90}")
print(f"Top 10 (preprocessing, model) combinations by min RMSECV")
print(f"{'_'*90}")
top10 = sweep_results_protein.sort_values("RMSECV_min").head(10).reset_index(drop=True)
for _, r in top10.iterrows():
    marker = " *" if r["rule_differs"] else "  "
    print(f"{marker}{r['model']:5s}  {r['preprocessing']:18s}  "
          f"RMSECV = {r['RMSECV_min']:.3f}  min = {r['params_min']}  "
          f"1-SE = {r['params_1se']}")
print("  (* = 1-SE rule picked a simpler model than minimum RMSECV)")

# Per-model: top 5 + full 21
for model in models_run:
    print(f"\n{'_'*90}\n{model} Top 5 preprocessings\n{'_'*90}")
    print(model_table(sweep_results_protein, model, top_n=5).to_string(index=False))

    print(f"\n{'_'*90}\n{model} All 21 preprocessings\n{'_'*90}")
    print(model_table(sweep_results_protein, model).to_string(index=False))

## CARS Wavelength Selection: Joint Preprocessing × Model Sweep
PROTEIN ONLY

In [ ]:
# CARS-based preprocessing × model sweep (protein)



# CARS hyperparameters 
CARS_N_SAMPLINGS = 50    # Monte Carlo sampling runs inside one CARS call
CARS_MAX_LVS     = 10    # max latent variables CARS considers
CARS_CV_FOLDS    = 10     # CARS's own internal CV (this is internal to CARS)

# accumulator for CARS sweep results
sweep_results_protein_cars = pd.DataFrame(
    columns=["preprocessing", "model", "RMSECV", "params", "n_bands_mean", "time_s"]
)

# helper: run CARS on a training matrix, return selected band indices
def run_cars_select(X_train, Y_train, seed=SEED):
    """Run CARS once and return the indices of selected wavelengths."""
    np.random.seed(seed)   # make the per-fold sweep selection reproducible
    out = competitive_adaptive_sampling(
        X=X_train,
        y=Y_train,
        max_components=CARS_MAX_LVS,
        folds=CARS_CV_FOLDS,
        iterations=CARS_N_SAMPLINGS,   # was n_sampling_runs
        adaptive_resampling=True,   # matches other cells
        preprocess="center",
        verbose=0,
    )
    return np.asarray(out["selected_variables"], dtype=int)

#LOOCV with CARS inside each fold 
def loocv_rmse_with_cars(X, Y, fold_masks, model_factory):
    preds = np.full(len(Y), np.nan)
    n_bands_list = []
    for test_mask in fold_masks.values():
        train_mask = ~test_mask
        # CARS fit on training brands only
        sel = run_cars_select(X[train_mask], Y[train_mask], seed=SEED)
        if len(sel) < 2:
            continue
        n_bands_list.append(len(sel))
        # fit model on CARS-selected bands of training data
        X_tr = X[train_mask][:, sel]
        X_te = X[test_mask][:, sel]
        max_safe = min(X_tr.shape[0] - 1, X_tr.shape[1])
        m = model_factory()
        if isinstance(m, Pipeline):
            for step in m.named_steps.values():
                if hasattr(step, "n_components"):
                    if step.n_components > max_safe:
                        step.n_components = max(1, max_safe)
        elif hasattr(m, "n_components"):
            if m.n_components > max_safe:
                m.n_components = max(1, max_safe)
        m.fit(X_tr, Y[train_mask])
        preds[test_mask] = m.predict(X_te).ravel()
    rmsecv = float(np.sqrt(np.nanmean((preds - Y) ** 2)))
    mean_bands = float(np.mean(n_bands_list)) if n_bands_list else float("nan")
    return rmsecv, mean_bands

print(f"  CARS hyperparameters: {CARS_N_SAMPLINGS} samplings, "
      f"{CARS_MAX_LVS} max LVs, {CARS_CV_FOLDS}-fold internal CV")
print(f"  Outer LOOCV: brand-level, {len(fold_masks)} folds")
print(f"  Random seed: {SEED}")

In [ ]:
# PLSR + CARS sweep over 21 preprocessings (protein, brand-level LOOCV)


def best_plsr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for n_lv in pls_lvs:
        if n_lv >= X.shape[0]:
            break
        r, n_b = loocv_rmse_with_cars(
            X, Y, fold_masks,
            lambda nlv=n_lv: PLSRegression(n_components=nlv, scale=False),
        )
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"n_components": n_lv}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_plsr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PLSR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_protein_cars = sweep_results_protein_cars[
    sweep_results_protein_cars["model"] != "PLSR"
]
sweep_results_protein_cars = pd.concat(
    [sweep_results_protein_cars, new], ignore_index=True
)

print(f"\nPLSR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# PCR + CARS sweep over 21 preprocessings (protein, brand-level LOOCV)


def best_pcr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for n_pc in pcr_pcs:
        if n_pc >= X.shape[0]:
            break
        def factory(np_=n_pc):
            return Pipeline([
                ("scale", StandardScaler(with_std=False)),
                ("pca",   PCA(n_components=np_, random_state=SEED)),
                ("lr",    LinearRegression()),
            ])
        r, n_b = loocv_rmse_with_cars(X, Y, fold_masks, factory)
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"n_components": n_pc}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_pcr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "PCR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_protein_cars = sweep_results_protein_cars[
    sweep_results_protein_cars["model"] != "PCR"
]
sweep_results_protein_cars = pd.concat(
    [sweep_results_protein_cars, new], ignore_index=True
)

print(f"\nPCR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# SVMR + CARS sweep over 21 preprocessings (protein, brand-level LOOCV)


def best_svmr_cars(X, Y):
    best = {"rmsecv": np.inf, "params": None, "n_bands": np.nan}
    for C, gamma in svr_grid:
        def factory(C_=C, g_=gamma):
            return Pipeline([
                ("scale", StandardScaler()),
                ("svr",   SVR(C=C_, gamma=g_, kernel="rbf")),
            ])
        r, n_b = loocv_rmse_with_cars(X, Y, fold_masks, factory)
        if r < best["rmsecv"]:
            best = {"rmsecv": r, "params": {"C": C, "gamma": gamma}, "n_bands": n_b}
    return best

rows = []
t0 = time.time()
for prep in preprocessings:
    X = X_matrices_train[prep]
    t_s = time.time()
    out = best_svmr_cars(X, Y_full)
    rows.append({
        "preprocessing": prep,
        "model":         "SVMR",
        "RMSECV":        out["rmsecv"],
        "params":        out["params"],
        "n_bands_mean":  out["n_bands"],
        "time_s":        round(time.time() - t_s, 2),
    })
    print(f"  {prep:18s}  RMSECV={out['rmsecv']:.3f}  "
          f"bands={out['n_bands']:.0f}  ({rows[-1]['time_s']:.1f}s)")

new = pd.DataFrame(rows)
sweep_results_protein_cars = sweep_results_protein_cars[
    sweep_results_protein_cars["model"] != "SVMR"
]
sweep_results_protein_cars = pd.concat(
    [sweep_results_protein_cars, new], ignore_index=True
)

print(f"\nSVMR+CARS sweep done in {(time.time()-t0)/60:.1f} min")
print(new.sort_values("RMSECV").head(10).to_string(index=False))

In [ ]:
# Print CARS sweep tables for the dissertation


# format helper: turn the params dict into a compact string
def fmt_params(params):
    if "n_components" in params:
        return str(params["n_components"])
    if "C" in params and "gamma" in params:
        # match the formatting used in your full-spectrum table
        return f"C={params['C']}, γ={params['gamma']:.0e}".replace("e-0", "e-")
    return str(params)


def fmt_prep(prep):
    if prep == "raw":
        return "Raw"
    if prep == "snv":
        return "SNV"
    if prep == "msc":
        return "MSC"
    parts = prep.split("_")
    out = parts[0].upper()                 
    out += f"({parts[1][1:]})"                      
    if len(parts) == 3:
        out += f"-{parts[2].upper()}"                
    return out


df = sweep_results_protein_cars.copy()
df["Preprocessing"]   = df["preprocessing"].apply(fmt_prep)
df["Hyperparameters"] = df["params"].apply(fmt_params)
df["Mean Bands"]      = df["n_bands_mean"].round(0).astype("Int64")
df["RMSECV (%)"]      = df["RMSECV"].round(3)

MODEL_ORDER = ["PLSR", "PCR", "SVMR"]

# top 5 per model 
print("_" * 90)
print("IN-TEXT TABLE: Top 5 (preprocessing, hyperparameters) per model")
print("_" * 90)
for m in MODEL_ORDER:
    sub = (df[df["model"] == m]
           .sort_values("RMSECV")
           .head(5)
           .reset_index(drop=True))
    sub.index = sub.index + 1
    sub.index.name = "Rank"
    print(f"\n{m}")
    print(sub[["Preprocessing", "Hyperparameters", "Mean Bands", "RMSECV (%)"]]
          .to_string())

# full 21 × 3 grid 
print("\n\n" + "_" * 90)
print("APPENDIX TABLE: Full grid (21 preprocessings × 3 models)")
print("_" * 90)
for m in MODEL_ORDER:
    sub = (df[df["model"] == m]
           .sort_values("RMSECV")
           .reset_index(drop=True))
    sub.index = sub.index + 1
    sub.index.name = "Rank"
    print(f"\n{m}")
    print(sub[["Preprocessing", "Hyperparameters", "Mean Bands", "RMSECV (%)"]]
          .to_string())

## External Prediction
PROTEIN ONLY

In [ ]:
# External prediction - Full spectral range




# derive rank-1 winners from sweep_results_protein (uses RMSECV_min rule)
sweep_winners_protein = {}
for m in sweep_results_protein["model"].unique():
    row = (sweep_results_protein[sweep_results_protein["model"] == m]
           .sort_values("RMSECV_min").iloc[0])
    sweep_winners_protein[m] = {
        "preprocessing": row["preprocessing"],
        "RMSECV":        row["RMSECV_min"],
        "params":        row["params_min"],
    }

print("Full-spectrum rank-1 winners:")
for m, w in sweep_winners_protein.items():
    print(f"  {m:5s}  prep={w['preprocessing']:18s}  "
          f"RMSECV={w['RMSECV']:.3f}  params={w['params']}")
print()

# model factories
def make_plsr(n_components):
    return PLSRegression(n_components=n_components, scale=False)

def make_pcr(n_components):
    return Pipeline([
        ("scale", StandardScaler(with_std=False)),
        ("pca",   PCA(n_components=n_components, random_state=SEED)),
        ("lr",    LinearRegression()),
    ])

def make_svmr(C, gamma):
    return Pipeline([
        ("scale", StandardScaler()),
        ("svr",   SVR(C=C, gamma=gamma, kernel="rbf")),
    ])

def evaluate(y_true, y_pred):
    rmsep = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    r2p   = float(r2_score(y_true, y_pred))
    bias  = float(np.mean(y_pred - y_true))
    return rmsep, r2p, bias

results_full = []
preds_full   = {}

for m_name, winner in sweep_winners_protein.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    if m_name == "PLSR":
        model = make_plsr(params["n_components"])
    elif m_name == "PCR":
        model = make_pcr(params["n_components"])
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr, Y_train_protein)
    y_pred = model.predict(X_va).ravel()
    preds_full[m_name] = y_pred

    rmsep, r2p, bias = evaluate(Y_val_protein, y_pred)
    results_full.append({
        "model":          m_name,
        "analysis":       "Full",
        "preprocessing":  prep,
        "params":         params,
        "n_bands":        X_tr.shape[1],
        "RMSECV":         winner["RMSECV"],
        "RMSEP":          rmsep,
        "R2p":            r2p,
        "bias":           bias,
    })
    print(f"  {m_name:5s}  RMSEP={rmsep:.3f}  R²p={r2p:.3f}  bias={bias:+.3f}")

results_full_df = pd.DataFrame(results_full)
print()
print(results_full_df[["model", "preprocessing", "n_bands",
                       "RMSECV", "RMSEP", "R2p", "bias"]].to_string(index=False))

In [ ]:
# external prediction on CARS-selected wavelengths (single fixed-seed run)


sweep_winners_protein_cars = {}
for m in sweep_results_protein_cars["model"].unique():
    row = (sweep_results_protein_cars[sweep_results_protein_cars["model"] == m].sort_values("RMSECV").iloc[0])
    sweep_winners_protein_cars[m] = {
        "preprocessing": row["preprocessing"],
        "RMSECV":        row["RMSECV"],
        "params":        row["params"],
        "n_bands_mean":  row["n_bands_mean"],
    }

print("CARS rank-1 winners (preprocessing per model):")
for m, w in sweep_winners_protein_cars.items():
    print(f"  {m:5s}  prep={w['preprocessing']:14s}  params={w['params']}")
print()

results_cars = []
preds_cars   = {}
cars_subsets = {}

for m_name, winner in sweep_winners_protein_cars.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    # one fixed-seed CARS run on the full calibration set
    sel = run_cars_select(X_tr, Y_train_protein, seed=SEED)
    cars_subsets[m_name] = sel
    print(f"  {m_name:5s}  prep={prep:14s}  bands={len(sel)} -> "
          f"{', '.join(f'{wavelengths_trimmed[b]:.0f}' for b in sorted(sel))} nm")

    X_tr_sel = X_tr[:, sel]
    X_va_sel = X_va[:, sel]

    if m_name == "PLSR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model = make_plsr(n_comp)
    elif m_name == "PCR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model = make_pcr(n_comp)
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr_sel, Y_train_protein)
    y_pred = model.predict(X_va_sel).ravel()
    preds_cars[m_name] = y_pred

    rmsep, r2p, bias = evaluate(Y_val_protein, y_pred)
    results_cars.append({
        "model":         m_name,
        "analysis":      "CARS",
        "preprocessing": prep,
        "params":        params,
        "n_bands":       len(sel),
        "RMSECV":        winner["RMSECV"],
        "RMSEP":         rmsep,
        "R2p":           r2p,
        "bias":          bias,
    })
    print(f"  -> RMSEP={rmsep:.3f}  R²p={r2p:.3f}  bias={bias:+.3f}\n")

results_cars_df = pd.DataFrame(results_cars)
print("=" * 70)
print(results_cars_df[["model", "preprocessing", "n_bands",
                       "RMSECV", "RMSEP", "R2p", "bias"]].to_string(index=False))

cars_subsets_protein = dict(cars_subsets)


In [ ]:
# External prediction summary - protein


combined = pd.concat([results_full_df, results_cars_df], ignore_index=True)
combined = combined.sort_values(["model", "analysis"]).reset_index(drop=True)

print("_" * 100)
print("External prediction summary: protein")
print("_" * 100)
print(combined[["model", "analysis", "preprocessing", "n_bands",
                "RMSECV", "RMSEP", "R2p", "bias"]].to_string(
    index=False,
    formatters={"RMSECV": "{:.3f}".format,
                "RMSEP":  "{:.3f}".format,
                "R2p":    "{:.3f}".format,
                "bias":   "{:+.3f}".format},
))

print()
print("_" * 100)
print("Per-bag predictions on external set (V/X/Z bags)")
print("_" * 100)
per_bag = pd.DataFrame({"bag": bag_order_val, "y_true": Y_val_protein})
for m in ["PLSR", "PCR", "SVMR"]:
    per_bag[f"{m}_Full"] = preds_full[m]
    per_bag[f"{m}_CARS"] = preds_cars[m]
print(per_bag.to_string(index=False, float_format="{:.3f}".format))

In [ ]:

# Per-brand RMSEP and bias for all 6 configurations
brand_of = [b[0] for b in bag_order_val]
rows = []
for m in ["PLSR", "PCR", "SVMR"]:
    for analysis, preds in [("Full", preds_full[m]), ("CARS", preds_cars[m])]:
        for brand in ["V", "X", "Z"]:
            mask = np.array([br == brand for br in brand_of])
            y_t = Y_val_protein[mask]
            y_p = preds[mask]
            rmsep = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
            bias  = float(np.mean(y_p - y_t))
            r2    = float(r2_score(y_t, y_p)) if mask.sum() >= 3 else float("nan")
            rows.append({"model": m, "analysis": analysis, "brand": brand,
                         "n_bags": int(mask.sum()),
                         "RMSEP": rmsep, "bias": bias, "R2p": r2})

per_brand = pd.DataFrame(rows)
print(per_brand.to_string(
    index=False,
    formatters={"RMSEP": "{:.3f}".format,
                "bias":  "{:+.3f}".format,
                "R2p":   "{:.3f}".format},
))

In [ ]:
# Complete NIR reporting metrics for all 6 configurations (protein)


# RPD reference: SD of the prediction set (Williams & Sobering 1996 convention)
sd_val = float(np.std(Y_val_protein, ddof=1))

# R2cv computed from sweep results: R2cv = 1 - (RMSECV^2) / Var(Y_cal)
var_cal = float(np.var(Y_train_protein, ddof=1))

rows = []

# Full-spectrum configurations
for m_name, winner in sweep_winners_protein.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    if m_name == "PLSR":
        model = make_plsr(params["n_components"])
    elif m_name == "PCR":
        model = make_pcr(params["n_components"])
    elif m_name == "SVMR":
        model = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr, Y_train_protein)
    y_cal_pred = model.predict(X_tr).ravel()
    y_val_pred = model.predict(X_va).ravel()

    rmsec  = float(np.sqrt(np.mean((y_cal_pred - Y_train_protein)**2)))
    r2c    = float(r2_score(Y_train_protein, y_cal_pred))
    rmsecv = winner["RMSECV"]
    r2cv   = 1.0 - (rmsecv**2) / var_cal
    rmsep  = float(np.sqrt(np.mean((y_val_pred - Y_val_protein)**2)))
    r2p    = float(r2_score(Y_val_protein, y_val_pred))
    bias   = float(np.mean(y_val_pred - Y_val_protein))
    rpd    = sd_val / rmsep

    rows.append({"model": m_name, "analysis": "Full",
                 "R2c": r2c, "RMSEC": rmsec,
                 "R2cv": r2cv, "RMSECV": rmsecv,
                 "R2p": r2p, "RMSEP": rmsep,
                 "bias": bias, "RPD": rpd})

# CARS configurations 
for m_name, winner in sweep_winners_protein_cars.items():
    prep   = winner["preprocessing"]
    params = winner["params"]
    X_tr   = X_matrices_train[prep]
    X_va   = X_matrices_val[prep]

    sel = cars_subsets[m_name]
    X_tr_sel = X_tr[:, sel]
    X_va_sel = X_va[:, sel]

    if m_name == "PLSR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model  = make_plsr(n_comp)
    elif m_name == "PCR":
        n_comp = min(params["n_components"], X_tr_sel.shape[1])
        model  = make_pcr(n_comp)
    elif m_name == "SVMR":
        model  = make_svmr(params["C"], params["gamma"])

    model.fit(X_tr_sel, Y_train_protein)
    y_cal_pred = model.predict(X_tr_sel).ravel()
    y_val_pred = model.predict(X_va_sel).ravel()

    rmsec  = float(np.sqrt(np.mean((y_cal_pred - Y_train_protein)**2)))
    r2c    = float(r2_score(Y_train_protein, y_cal_pred))
    rmsecv = winner["RMSECV"]
    r2cv   = 1.0 - (rmsecv**2) / var_cal
    rmsep  = float(np.sqrt(np.mean((y_val_pred - Y_val_protein)**2)))
    r2p    = float(r2_score(Y_val_protein, y_val_pred))
    bias   = float(np.mean(y_val_pred - Y_val_protein))
    rpd    = sd_val / rmsep

    rows.append({"model": m_name, "analysis": "CARS",
                 "R2c": r2c, "RMSEC": rmsec,
                 "R2cv": r2cv, "RMSECV": rmsecv,
                 "R2p": r2p, "RMSEP": rmsep,
                 "bias": bias, "RPD": rpd})

metrics_df = pd.DataFrame(rows).sort_values(["model","analysis"]).reset_index(drop=True)

print("_" * 110)
print("Complete NIR metrics: protein")
print(f"SD(Y_val) = {sd_val:.3f} %  |  SD(Y_cal) = {np.std(Y_train_protein, ddof=1):.3f} %")
print("_" * 110)
print(metrics_df.to_string(
    index=False,
    formatters={"R2c":    "{:.3f}".format,
                "RMSEC":  "{:.3f}".format,
                "R2cv":   "{:.3f}".format,
                "RMSECV": "{:.3f}".format,
                "R2p":    "{:.3f}".format,
                "RMSEP":  "{:.3f}".format,
                "bias":   "{:+.3f}".format,
                "RPD":    "{:.2f}".format},
))

In [ ]:
# CARS-selected wavelengths for moisture and protein


def wl_axis_for(prep):
    n_features = X_matrices_train[prep].shape[1]
    if len(wavelengths_trimmed) == n_features:
        return wavelengths_trimmed
    if len(wavelengths) == n_features:
        return wavelengths
    return None

targets = {
    "MOISTURE": (cars_subsets_moisture, sweep_winners_moisture_cars),
    "PROTEIN":  (cars_subsets_protein,  sweep_winners_protein_cars),
}

for target, (subsets, winners) in targets.items():
    print("_" * 60)
    print(f"CARS-selected wavelengths, {target}")
    print("_" * 60)
    for m_name, sel in subsets.items():
        prep = winners[m_name]["preprocessing"]
        sel = np.asarray(sel, dtype=int)
        axis = wl_axis_for(prep)
        if axis is None:
            print(f"\n{m_name}  ({len(sel)} bands, prep={prep})")
            print(f"  no wavelength axis matches feature count; showing indices")
            print(f"  {sorted(sel.tolist())}")
            continue
        nm = sorted(axis[sel].tolist())
        print(f"\n{m_name}  ({len(sel)} bands, prep={prep})")
        print(f"  {', '.join(f'{w:.0f}' for w in nm)} nm")
        print(f"  range: {min(nm):.0f}-{max(nm):.0f} nm")
    print()


# Visualization Maps

In [ ]:
# pixel-level distribution maps for moisture and protein
# applies each target's full-spectrum PLSR winner to every pixel of selected

# pick 5 bags per target spanning the range
def pick_range_spanning_bags(metadata, target_col, n_samples=5):
    sorted_bags = metadata.sort_values(target_col).index.tolist()
    sorted_values = metadata.loc[sorted_bags, target_col].tolist()
    idx = np.linspace(0, len(sorted_bags) - 1, n_samples).astype(int)
    return [(sorted_bags[i], sorted_values[i]) for i in idx]


def snv_pixel(spectra_2d):
    means = spectra_2d.mean(axis=1, keepdims=True)
    stds  = spectra_2d.std(axis=1, keepdims=True)
    stds[stds == 0] = 1
    return (spectra_2d - means) / stds


def preprocess_pixels_moisture(cube_pixels):
    # matches the moisture full-spectrum winner: SG1 (window 11) then MSC
    sg1 = savgol_filter(cube_pixels, window_length=11, polyorder=2, deriv=1, axis=1)
    sg1_msc = np.array([apply_msc(spec, reference) for spec in sg1])
    return sg1_msc[:, 8:-6]


def preprocess_pixels_protein(cube_pixels):
    # matches the protein full-spectrum winner: SG2 (window 9) then SNV
    sg2 = savgol_filter(cube_pixels, window_length=9, polyorder=2, deriv=2, axis=1)
    sg2_snv = snv_pixel(sg2)
    return sg2_snv[:, 8:-6]


def predict_map(bag, replicate_id, model, preprocess_fn, erosion_radius=20):
    sample_id = f"S{replicate_id}{bag}"
    if "masks" in globals() and sample_id not in masks:
        return None, None
    cube = load_cube_on_demand(sample_id, "masked")
    Y_dim, X_dim, n_bands = cube.shape
    # erode the mask to drop edge pixels
    interior = ~np.isnan(cube[:, :, 0])
    interior = binary_erosion(interior, footprint=disk(erosion_radius))
    cube_int = cube.copy()
    cube_int[~interior] = np.nan
    pixels = cube_int.reshape(-1, n_bands)
    valid = ~np.isnan(pixels).any(axis=1)
    pp = preprocess_fn(pixels[valid])
    preds = model.predict(pp).ravel()
    flat = np.full(pixels.shape[0], np.nan)
    flat[valid] = preds
    return flat.reshape(Y_dim, X_dim), preds.mean()


def crop_to_content(pmap):
    # trim the nan border so the sample fills the panel
    rows = np.where(~np.isnan(pmap).all(axis=1))[0]
    cols = np.where(~np.isnan(pmap).all(axis=0))[0]
    if len(rows) == 0 or len(cols) == 0:
        return pmap
    return pmap[rows.min():rows.max()+1, cols.min():cols.max()+1]


def plot_distribution_maps(selected_samples, model, preprocess_fn, target_label,
                           target_unit, target_format, replicate_id=1,
                           color_percentiles=(5, 95)):
    selected_samples = sorted(selected_samples, key=lambda t: t[1])  # low to high
    n = len(selected_samples)

    maps, pred_means = [], []
    for bag, ref_val in selected_samples:
        pmap, pmean = predict_map(bag, replicate_id, model, preprocess_fn)
        maps.append(pmap); pred_means.append(pmean)

    all_valid = np.concatenate([m[~np.isnan(m)] for m in maps if m is not None])
    vmin, vmax = np.percentile(all_valid, color_percentiles)

    cmap = plt.cm.jet.copy()
    cmap.set_bad(alpha=0)  # transparent background

    fig, axes = plt.subplots(1, n, figsize=(3.0 * n, 3.6))
    if n == 1:
        axes = [axes]

    im = None
    for ax, (bag, ref_val), pmap, pmean in zip(axes, selected_samples, maps, pred_means):
        if pmap is None:
            ax.text(0.5, 0.5, f"{bag}\nmissing", ha="center", va="center")
            ax.axis("off")
            continue
        pmap = crop_to_content(pmap)
        im = ax.imshow(pmap, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_title(f"{bag}", fontsize=13, pad=8)
        ax.set_xlabel(f"Ref: {ref_val:{target_format}}\nPred mean: {pmean:{target_format}}",
                      fontsize=10, labelpad=8)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)

    fig.colorbar(im, ax=axes, orientation="vertical", pad=0.02,
                 fraction=0.025, label=f"{target_label} ({target_unit})")
    fig.suptitle(f"{target_label} distribution maps", fontsize=14, y=1.04)
    plt.show()


# train the full-spectrum PLSR winners (linear coefficients apply per pixel)
print("training PLSR models for visualization...")
X_train_moisture = X_matrices_train["sg1_w11_msc"]
n_lv_moisture = cv_results_moisture["sg1_w11_msc"]["n_lv_min"]
plsr_moisture = PLSRegression(n_components=n_lv_moisture, scale=False)
plsr_moisture.fit(X_train_moisture, Y_train_moisture)
print(f"  moisture PLSR: {n_lv_moisture} LVs on {X_train_moisture.shape}")

X_train_protein = X_matrices_train["sg2_w9_snv"]
n_lv_protein = cv_results_protein["sg2_w9_snv"]["n_lv_min"]
plsr_protein = PLSRegression(n_components=n_lv_protein, scale=False)
plsr_protein.fit(X_train_protein, Y_train_protein)
print(f"  protein PLSR:  {n_lv_protein} LVs on {X_train_protein.shape}")


# moisture maps
selected_for_moisture = pick_range_spanning_bags(metadata_train, "moisture", n_samples=5)
print("\nmoisture samples (low to high):")
for bag, val in sorted(selected_for_moisture, key=lambda t: t[1]):
    print(f"  {bag}: {val:.2f}%")
plot_distribution_maps(selected_for_moisture, plsr_moisture,
                       preprocess_pixels_moisture, "Moisture", "%", ".2f")


# protein maps
selected_for_protein = pick_range_spanning_bags(metadata_train, "protein", n_samples=5)
print("\nprotein samples (low to high):")
for bag, val in sorted(selected_for_protein, key=lambda t: t[1]):
    print(f"  {bag}: {val:.2f}%")
plot_distribution_maps(selected_for_protein, plsr_protein,
                       preprocess_pixels_protein, "Protein", "g/100 g", ".2f")
